# CS-781 Capstone — Phase 1: CALM — a Continuous Autoregressive Language Model, trained from scratch

This notebook trains and submits ONE architecture end-to-end on Kaggle:
1. Get the legal text data and reuse the saved from-scratch BPE tokenizer
2. **Stage 1:** train an autoencoder that squeezes every 4 consecutive tokens into ONE vector of 128 numbers
3. **Stage 2:** train a transformer that works on those vectors — one position per 4 tokens — and predicts the *next* vector
4. Generate the 128-word continuations for the test set (chunk by chunk, with a KV cache)
5. Package `submission.zip` to upload to Codabench

**Workflow:** run it with `QUICK_TEST = True` first (Settings cell) to confirm everything works on a tiny slice, then set
`QUICK_TEST = False` and run it for real. Both stages checkpoint to the Hugging Face bucket and resume (also across sessions / accounts).

**Before running, in the panel on the right:** Accelerator → a GPU (T4); Internet → ON.

---
**What is new here, in one paragraph.** A normal language model predicts one *token* at a time with a softmax over the 40,000-word
vocabulary. CALM (Shao et al., 2025, arXiv 2510.27688) predicts one *chunk of 4 tokens* at a time, as a single continuous vector. So the
backbone runs 4× fewer steps, and the 40,000-way output layer disappears. The price: there is no probability for the next token, so
the model is trained with the **energy score** (a likelihood-free loss) instead of cross-entropy, and the text is recovered by decoding the
predicted vector with the Stage-1 autoencoder.

**Honest caveats** (also for the report): the paper's smallest model is 371M parameters trained on ~215B tokens; we are at ~175M parameters
and ~360M tokens, so there is no evidence it works at our scale. The paper's claim is better *efficiency* (quality per FLOP), not better
quality at equal parameters. This notebook is our own small-scale reimplementation; where it differs from the paper (for example the
optional `USE_LOCAL_MIX` layer, the voting decoder, the warmup + cosine learning-rate schedule) it says so in the code.

## 1. Settings

In [1]:
# ----------------------------------------------------------------
# QUICK_TEST lets you sanity-check the whole notebook in a few
# minutes before committing GPU time to a real run.
#
#   1. First run: leave QUICK_TEST = True. Run every cell top to
#      bottom. It trains BOTH stages on a tiny slice of data, just to
#      prove nothing crashes and a submission.zip comes out the other end.
#   2. Once that works: set QUICK_TEST = False and re-run every cell.
#      This is the real run: Stage 1 (autoencoder, stops early once it is
#      accurate enough) and then Stage 2 (the language model).
# ----------------------------------------------------------------
QUICK_TEST = False

if QUICK_TEST:
    MAX_WORDS_FOR_TRAINING = 200_000   # a small slice, just to test the pipeline
    AE_WORDS = 200_000
    NUM_EPOCHS = 1
else:
    MAX_WORDS_FOR_TRAINING = 240_000_000   # words the LANGUAGE MODEL (Stage 2) trains on; the speed probe cell in Section 9 prints the projected hours
    AE_WORDS = 120_000_000                 # words the AUTOENCODER (Stage 1) may train on (it stops earlier if it reaches AE_TARGET_ACC)
    NUM_EPOCHS = 1
PER_SOURCE_WORDS = MAX_WORDS_FOR_TRAINING // 4   # equal quota from each of the 4 sources (acts / high courts / supreme court / tribunals)

VOCAB_SIZE = 40000        # BPE subword vocabulary size (the saved from-scratch Assignment-1 tokenizer)
CONTEXT_LEN = 2048        # TOKENS per training window = CONTEXT_LEN / CHUNK_K = 512 backbone positions
TEMPERATURE = 0.0         # (unused by CALM; kept so the shared cells that print it still run) -- see GEN_* knobs in Section 13

# ----------------------------------------------------------------
# CALM = Continuous Autoregressive Language Model (Shao et al., 2025, arXiv 2510.27688), our small-scale version.
# Stage 1: an autoencoder squeezes every CHUNK_K consecutive tokens into ONE vector of LATENT_DIM numbers.
# Stage 2: a transformer reads one position per chunk and predicts the NEXT chunk's vector -- so it takes
#          CHUNK_K times fewer steps, and there is no vocabulary-sized softmax anywhere in it.
# ----------------------------------------------------------------
CHUNK_K = 4               # tokens per chunk. CONTEXT_LEN and the prompt window must be multiples of this.
LATENT_DIM = 128          # numbers per chunk vector (paper: 128 for K=4)

# --- Stage 1: autoencoder -------------------------------------------------------------------------
AE_D = 512                # the autoencoder's working width (its own embedding table is VOCAB_SIZE x AE_D, tied with its output layer)
AE_FF = 2048              # width of its feed-forward blocks (4 x AE_D)
AE_KL_WEIGHT = 1e-3       # weight of the KL term (keeps latents near N(0,1); small, so reconstruction wins)
AE_KL_CLIP = 0.5          # per-dimension "free bits": a dimension is not penalised until its KL exceeds this
AE_LATENT_DROPOUT = 0.15  # dropout on the latent before decoding -- robustness to imperfect guesses
AE_TOKEN_MASK = 0.15      # share of INPUT tokens blanked while training -- same purpose (targets stay the real tokens)
AE_BATCH_CHUNKS = 1024    # chunks (of CHUNK_K tokens) per training step; each costs a V x AE_D logits matrix, so lower this if you run out of GPU memory
AE_LR = 5e-4
AE_WARMUP_STEPS = 500
AE_TARGET_ACC = 0.999     # stop Stage 1 as soon as the held-out per-token reconstruction accuracy (decoding a SAMPLED latent) reaches this
AE_MIN_STEPS = 3000       # ...but never before this many steps
AE_EVAL_EVERY = 1000      # steps between held-out accuracy checks (and checkpoints)

# --- Stage 2: the language model -------------------------------------------------------------------
EMB_D = 768               # width of the backbone's own input embedding table (VOCAB_SIZE x EMB_D); the K token vectors are flattened to CHUNK_K*EMB_D
D_MODEL = 768             # backbone width
N_LAYERS = 12             # backbone depth
N_HEADS = 12              # query heads (head dim = D_MODEL / N_HEADS = 64)
GQA_GROUP_SIZE = 2        # query heads sharing one key/value head -> N_HEADS / 2 = 6 key/value heads
D_FF = 6144               # FFN width per layer. 3072 = 4x (about 147M params in total), 4608 = 6x (about 175M), 6144 = 8x (about 203M)
USE_LOCAL_MIX = False     # True adds a tiny self-attention layer over the CHUNK_K tokens of a chunk before they are merged (our own addition; not in the paper)
HEAD_D = 512              # width of the generative (energy) head
HEAD_LAYERS = 3           # residual FFN blocks in the head
NOISE_DIM = 128           # size of the random noise vector fed to the head
ENERGY_SAMPLES = 8        # N: guesses the head makes per position during training (paper: 8). Cheap, but they cost head compute x N
ENERGY_TARGETS = 100      # M: samples drawn from the autoencoder's posterior as the "real" targets in the loss (paper: 100)

BATCH_SIZE = 8            # token windows (of CONTEXT_LEN tokens) per Stage-2 step
LEARNING_RATE = 3e-4      # peak Stage-2 learning rate (linear warmup, then cosine decay to MIN_LR_FRACTION of it)
LR_WARMUP_STEPS = 1000
MIN_LR_FRACTION = 0.1
GRAD_CLIP = 1.0           # clip the gradient norm at this value (0 = off)

# ----------------------------------------------------------------
# Research knobs shared with the other notebooks
# ----------------------------------------------------------------
NORM_TYPE = "layernorm"   # "layernorm" | "rmsnorm"  (CALM uses pre-norm blocks only)
OPTIMIZER_TYPE = "adamw"  # "adamw" | "muon"
assert NORM_TYPE in ("layernorm", "rmsnorm"), "CALM supports NORM_TYPE 'layernorm' or 'rmsnorm'"
assert CONTEXT_LEN % CHUNK_K == 0, "CONTEXT_LEN must be a multiple of CHUNK_K"
NORM_POSITION = "pre"

RUN_TAG = "calm240Mw_1ep"
TOKENIZER_RUN_TAG = "160Mw_1ep"   # the tokenizer is shared with the 160M-word runs (same corpus family, same BPE), so it is loaded, not retrained
TOKENIZER_WORDS = 160_000_000     # the word count in that saved tokenizer's file name (the corpus here is larger, but the tokenizer is reused as is)
REQUIRE_SAVED_TOKENIZER = True    # full run: stop with a clear message if that saved tokenizer is not in the bucket, instead of silently re-training it
mode_tag = "quicktest" if QUICK_TEST else "full"

## 2. Make sure the working folder exists (and a note on Kaggle persistence)

Checkpoints go to `/kaggle/working`. That folder survives you re-running
cells, and even a "Restart session", within the same notebook edit session.

If you come back on a **completely new day / new session** and want to
resume: click **Save Version** before you stop working. Next time, use
**Add Input → Notebook Output Files → (this notebook's last saved version)**
to pull the checkpoint back in, then copy it into `/kaggle/working` before
re-running the training cell.

In [2]:
import os
os.makedirs("/kaggle/working", exist_ok=True)
print("Working folder ready:", os.path.exists("/kaggle/working"))

Working folder ready: True


## 3. One-time setup, then install and import what we need

**The dataset is GATED** -- before this notebook can download anything, do
this once per Kaggle **account** (each of your two accounts needs its own):

1. While logged into Hugging Face, open
   https://huggingface.co/datasets/Exploration-Lab/CS-781-Capstone and
   accept the dataset's access conditions (there's a form to agree to on
   that page -- without this, downloads below get an auth error).
2. Create a **read** access token at https://huggingface.co/settings/tokens.
3. In this Kaggle notebook: **Add-ons → Secrets → Add a new secret**, name
   it exactly `HF`, and paste the token as its value.

Do this before running the cell below -- it logs in using that secret.

In [3]:
!pip install -q -U "huggingface_hub>=1.5"   # storage buckets need huggingface_hub >= 1.5
import json
import os
import zipfile
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import hf_hub_download, login
from kaggle_secrets import UserSecretsClient

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

user_secrets = UserSecretsClient()

def _clean_secret(name):
    """Kaggle secrets pasted from a web page can carry invisible/non-ASCII characters (e.g. a middle dot) that break HTTP headers.
    Keep only the first hf_... token. Never prints the token itself."""
    import re
    raw = user_secrets.get_secret(name)
    m = re.search(r"hf_[A-Za-z0-9]+", raw)
    tok = m.group(0) if m else "".join(ch for ch in raw if 32 < ord(ch) < 127)
    if tok != raw:
        print(f"WARNING: Kaggle secret {name} had extra/non-ASCII characters (length {len(raw)} -> {len(tok)}); cleaned. Re-save the secret to fix permanently.")
    return tok

HF_TOKEN = _clean_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("Hugging Face login successful.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.0/838.0 kB 11.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 2.1.1 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.1.1 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 2.1.1 which is incompatible.
transformers 5.16.1 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 2.1.1 which is incompatible.
gradio-client 2.6.1 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.1.1 which is incompatible.
tokenizers 0.23.1 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.1.1 which is incompatible.
Using device: cuda


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


## 3b. Sync checkpoints with a Hugging Face model repo

Instead of manually downloading/uploading `.pt` files between Kaggle
sessions or accounts, checkpoints are pushed to a Hugging Face **model**
repo after every epoch, and pulled back down automatically if this session
doesn't already have a local one. The filename already encodes the full
config (architecture, QUICK_TEST mode, norm/attention/optimizer settings --
see the Settings cell), so the repo doubles as a self-documenting archive
of every run you've tried -- handy when writing up the report.

**One-time setup (separate from the read-only `HF` secret used for the
gated dataset):**
1. Create a Hugging Face **model** repo (e.g. at
   huggingface.co/new -> "Model"), private or public, your choice.
2. Create a **write**-scoped access token at
   https://huggingface.co/settings/tokens.
3. In this Kaggle notebook: **Add-ons -> Secrets -> Add a new secret**,
   name it exactly `HF_WRITE`, and paste that token as its value. (Keep
   the existing `HF` secret as-is -- that one stays read-only, just for
   downloading the dataset.)

In [4]:
from huggingface_hub import HfApi, create_repo, create_bucket, batch_bucket_files, download_bucket_files, list_bucket_tree, hf_hub_download
import shutil

HF_WRITE_TOKEN = _clean_secret("HF_WRITE")
hf_api = HfApi(token=HF_WRITE_TOKEN)

# ---- Where models live: a Hugging Face STORAGE BUCKET (not a repo) -------------------------------------------
# A repo is a git repository: every overwritten checkpoint stays in its history, so a few 3 GB models quickly
# cost tens of GB. A bucket is plain mutable storage: overwriting a file REPLACES it (no history), so each model
# costs exactly its own size. Needs huggingface_hub >= 1.5 (installed in Section 3).
HF_BUCKET = "kavincg23/cs781-models"
HF_CHECKPOINT_REPO = "kavincg23/cs781-checkpoints"   # the OLD repo: only READ, as a fallback for models not moved yet
create_bucket(HF_BUCKET, private=True, exist_ok=True, token=HF_WRITE_TOKEN)


def list_model_files():
    """Names of every file in the bucket (empty list if the bucket cannot be listed)."""
    try:
        return [f.path for f in list_bucket_tree(HF_BUCKET, recursive=True, token=HF_WRITE_TOKEN)
                if getattr(f, "type", "file") == "file"]
    except Exception as e:
        print(f"(could not list the bucket: {type(e).__name__})")
        return []


def upload_model_file(local_path, name=None, quiet=False):
    """Upload local_path to the bucket as `name` (default: its own filename). Replaces any older copy."""
    name = name or os.path.basename(local_path)
    batch_bucket_files(HF_BUCKET, add=[(local_path, name)], token=HF_WRITE_TOKEN)
    if not quiet:
        print(f"Uploaded {name} to bucket {HF_BUCKET}")


def fetch_model_file(name, local_path=None):
    """Download `name` from the bucket (falling back to the old repo) to local_path; returns the local path.
    Raises FileNotFoundError if it is in neither place."""
    local_path = local_path or f"/kaggle/working/{name}"
    if os.path.exists(local_path):
        return local_path
    part = local_path + ".part"
    try:
        download_bucket_files(HF_BUCKET, files=[(name, part)], raise_on_missing_files=True, token=HF_WRITE_TOKEN)
        if not os.path.exists(part):
            raise FileNotFoundError(name)
        os.replace(part, local_path)
        return local_path
    except Exception:
        if os.path.exists(part):
            os.remove(part)
    try:   # fallback: the old repo (read-only)
        p = hf_hub_download(repo_id=HF_CHECKPOINT_REPO, filename=name, repo_type="model", token=HF_WRITE_TOKEN)
        shutil.copy(p, local_path)
        print(f"(found {name} only in the old repo {HF_CHECKPOINT_REPO}; new saves go to the bucket)")
        return local_path
    except Exception:
        raise FileNotFoundError(f"{name} is in neither the bucket {HF_BUCKET} nor the repo {HF_CHECKPOINT_REPO}")


def push_checkpoint_to_hub(local_path, quiet=False):
    """Upload a checkpoint, the tokenizer, or the run log to the bucket under its own filename (which already
    encodes the full config -- see Settings). Each upload REPLACES the previous version, and nothing is kept."""
    upload_model_file(local_path, quiet=quiet)


def pull_from_hub(local_path):
    """Quietly fetch local_path's file (by basename) if it is not on disk yet. True if it exists afterwards."""
    if os.path.exists(local_path):
        return True
    try:
        fetch_model_file(os.path.basename(local_path), local_path)
        print(f"Pulled {os.path.basename(local_path)} from the Hub")
        return True
    except Exception:
        return False


def try_pull_checkpoint_from_hub(local_path):
    """If no LOCAL checkpoint exists yet (fresh Kaggle session / other account), fetch a matching one."""
    if os.path.exists(local_path):
        return
    try:
        fetch_model_file(os.path.basename(local_path), local_path)
        print(f"Pulled matching checkpoint from the Hub: {os.path.basename(local_path)}")
    except Exception as e:
        print(f"No matching checkpoint on the Hub yet ({type(e).__name__}) -- starting fresh.")


print(f"Model storage configured -> bucket {HF_BUCKET}")

# ---- Run log: a running record of what this project did, for the final report ----
# Every important step appends one JSON line (time, what happened, the numbers) to
# RUN_LOG_PATH. The file is pushed to the Hub together with the checkpoints and pulled
# back at the start of a new session, so a run that spans several sessions (or both
# Kaggle accounts) keeps ONE continuous log. Read it with: print(open(RUN_LOG_PATH).read())
import time as _time, datetime as _dt
RUN_LOG_PATH = f"/kaggle/working/run_log_gqa_kvcache_{RUN_TAG}_{mode_tag}.jsonl"
pull_from_hub(RUN_LOG_PATH)


def log_event(kind, **fields):
    """Append one event to the run log (never raises: logging must not stop a training run)."""
    try:
        record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}
        with open(RUN_LOG_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps(record, default=str) + "\n")
    except Exception as e:
        print(f"(run log write failed: {type(e).__name__})")


def sync_run_log():
    try:
        push_checkpoint_to_hub(RUN_LOG_PATH, quiet=True)
    except Exception as e:
        print(f"(run log push failed: {type(e).__name__})")


log_event("session_start", quick_test=QUICK_TEST, vocab_size=VOCAB_SIZE, context_len=CONTEXT_LEN,
          batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, num_epochs=NUM_EPOCHS,
          max_words=MAX_WORDS_FOR_TRAINING, norm=f"{NORM_TYPE}/{NORM_POSITION}", optimizer=OPTIMIZER_TYPE,
          device=device, gpu=(torch.cuda.get_device_name(0) if device == "cuda" else None))
print(f"Run log: {RUN_LOG_PATH}")

Model storage configured -> bucket kavincg23/cs781-models
Run log: /kaggle/working/run_log_gqa_kvcache_calm240Mw_1ep_full.jsonl


/tmp/ipykernel_23/2161731718.py:103: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


## 4. Download the data files from Hugging Face

This dataset has no `datasets.load_dataset(...)` config -- it's raw JSON
files, one per source, downloaded directly (the same way the official
starter notebook does it). Make sure **Internet** is turned ON in the
settings panel on the right.

**Important:** the 4 train files together hold the *entire* >4.5B-token
corpus. `json.load()`-ing even one of them fully into memory is enough to
exceed Kaggle's RAM and crash the notebook ("tried to allocate more memory
than is available") -- and that happens regardless of `MAX_WORDS_FOR_TRAINING`,
since that setting only limits how many words we *use*, not how much we'd
have loaded first. So instead of `json.load`, we use `ijson` to stream
through each file's JSON array one document at a time, only ever holding a
handful of documents in memory. The print statement below just peeks at
the first document, to let you SEE the real shape of the data.

In [5]:
!pip install -q ijson
import ijson

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_high_courts.json",
    "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Phase_1_Pretraining/Train/train_tribunals.json",
]
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

first_train_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=TRAIN_FILES[0],
                                    repo_type="dataset", token=HF_TOKEN)
with open(first_train_path, "r", encoding="utf-8") as f:
    preview_record = next(ijson.items(f, "item"))
print("Example document (from the first train file):")
print(preview_record)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.9/149.9 kB 3.4 MB/s eta 0:00:00


Phase_1_Pretraining/Train/train_acts.jso(…): reconstructing file:   0%|          |  0.00B /  729MB            

Phase_1_Pretraining/Train/train_acts.jso(…): downloading bytes:           |  0.00B            

Example document (from the first train file):
{'title': 'Bangalore City Planning Area Zonal Regulations (Amendment and Validation) Act, 1996', 'category': 'Greater Bengaluru City Corporation - Act', 'year': 1972, 'url': 'https://indiankanoon.org/doc/179573747/', 'text': 'Bangalore City Planning Area Zonal Regulations (Amendment and Validation) Act, 1996 (Karnataka Act No. 2 of 1996) Statements of objects and Reasons - Act 2 of 1996. - The Bangalore City Planning Area Zonal Regulations 1972, which was in vogue till 1984, provided for construction of five floors including ground floor. The Corporation of the City of Bangalore granted permission for construction of buildings beyond five floors in certain cases. In respect of some of the buildings the permission granted by the Corporation contrary to the Zonal Regulations then in force was questioned in the High Court and the Division Bench of the High Court held that the permission granted in violation of the Zonal Regulation was illegal.

### Check the field name above

Look at the example printed above -- it should be a dict like
`{'text': '...'}` (confirmed from the official starter notebook). Set
`TEXT_FIELD` below to whatever key actually holds the legal text --
change it if the printed example above used a different name.

In [6]:
TEXT_FIELD = "text"  # <-- change this if needed, based on what you saw above

### Download and load the test prompts (the ones we must generate continuations for)

`test.json` is a single JSON object keyed by ID (`{"1": {"text": "..."}, ...}`),
not a list -- that's already the exact shape Codabench wants predictions in.

In [7]:
test_file_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=TEST_FILE,
                                  repo_type="dataset", token=HF_TOKEN)
with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

test_prompts = {str(doc_id): record[TEXT_FIELD] for doc_id, record in test_data.items()}

print(f"Loaded {len(test_prompts)} test prompts")
first_id = list(test_prompts.keys())[0]
print(f"Example prompt (id={first_id}), word count = {len(test_prompts[first_id].split())}")

test.json:   0%|          | 0.00/3.03M [00:00<?, ?B/s]

Loaded 1000 test prompts
Example prompt (id=1), word count = 512


### Stream through the training files, collecting words until we have enough

This walks through `TRAIN_FILES` one document at a time (still via `ijson`,
never loading a full file into memory) and stops as soon as it has collected
`MAX_WORDS_FOR_TRAINING` words -- so a `QUICK_TEST` run only ever reads a
small slice of the corpus off disk, not all >4.5B tokens of it.

In [8]:
import array, random, time
SAMPLING_SEED = 42      # fixed seed: the same documents are chosen on every run
# Whole documents held out for EVALUATION (words per source; never trained on). The Supreme Court set is
# the largest because the test prompts look like Supreme Court text.
HELDOUT_WORDS = {'acts': 500000, 'high_courts': 500000, 'supreme_court': 2000000, 'tribunals': 500000}
HELDOUT_MIN_DOC_WORDS = 640       # long enough for a 512-word prompt + 128-word continuation, like the test examples
HELDOUT_MAX_WORDS_PER_DOC = 6000  # cap so a few enormous documents do not dominate
HELDOUT_FILES = {}                # source name -> file of held-out documents (one per line); filled below
print("Streaming through the training files, collecting words until we have enough...")

# Words are written straight to a FILE as we stream through the sources,
# instead of being collected into one giant Python list/string first. The
# BPE tokenizer below trains directly from this file (it streams it from
# disk in blocks) rather than us handing it one huge in-memory
# string -- this is what lets tokenizer training handle a large word count
# without running out of RAM (building the string in Python first was the
# actual bottleneck, not the tokenizer trainer itself).
CORPUS_FILE = "/kaggle/working/bpe_training_corpus.txt"

if QUICK_TEST:
    # fast pipeline check -- just fill up to MAX_WORDS_FOR_TRAINING from
    # whichever source(s) come first, same as before, just writing to the
    # corpus file instead of a Python list.
    word_count = 0
    with open(CORPUS_FILE, "w", encoding="utf-8") as out_f:
        for filename in TRAIN_FILES:
            if word_count >= MAX_WORDS_FOR_TRAINING:
                break
            local_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=filename,
                                          repo_type="dataset", token=HF_TOKEN)
            with open(local_path, "r", encoding="utf-8") as f:
                for record in ijson.items(f, "item"):
                    doc_words = record[TEXT_FIELD].split()
                    remaining = MAX_WORDS_FOR_TRAINING - word_count
                    if remaining <= 0:
                        break
                    if len(doc_words) > remaining:
                        doc_words = doc_words[:remaining]
                    out_f.write(" ".join(doc_words) + " ")
                    word_count += len(doc_words)
                    if word_count >= MAX_WORDS_FOR_TRAINING:
                        break
            print(f"  ...after {filename}: {word_count} words collected so far")
else:
    # Full run: pull a FIXED QUOTA from EACH of the 4 sources, instead of
    # filling up sequentially (which would silently train on only the
    # first source(s)). This is the "data mixing across sources"
    # experiment the brief expects -- change the quotas below to try a
    # different mix, and document what you tried.
    # Within each source the documents are a SEEDED RANDOM sample of the whole file
    # (not the first documents in it), so a file that happens to be sorted by court,
    # year, etc. cannot bias what the model sees. QUICK_TEST still takes the first documents.
    SOURCE_WORD_QUOTAS = {
    "Phase_1_Pretraining/Train/train_acts.json": PER_SOURCE_WORDS,
    "Phase_1_Pretraining/Train/train_high_courts.json": PER_SOURCE_WORDS,
    "Phase_1_Pretraining/Train/train_supreme_court.json": PER_SOURCE_WORDS,
    "Phase_1_Pretraining/Train/train_tribunals.json": PER_SOURCE_WORDS,
    }
    word_count = 0
    with open(CORPUS_FILE, "w", encoding="utf-8") as out_f:
        for filename in TRAIN_FILES:
            quota = SOURCE_WORD_QUOTAS.get(filename, 0)
            if quota <= 0:
                continue
            short = os.path.basename(filename)[len("train_"):-len(".json")]
            local_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=filename,
                                          repo_type="dataset", token=HF_TOKEN)

            # PASS 1: count the words of every document (the text itself is thrown
            # away immediately, so memory stays tiny even for the multi-GB files).
            t0 = time.time()
            doc_lengths = array.array("l")
            with open(local_path, "r", encoding="utf-8") as f:
                for record in ijson.items(f, "item"):
                    doc_lengths.append(len(record[TEXT_FIELD].split()))
            print(f"  {filename}: {len(doc_lengths):,} documents, {sum(doc_lengths):,} words "
                  f"(counted in {time.time() - t0:.0f}s)")

            # A seeded random order over ALL documents of the file: every document -- and
            # so every word -- is equally likely to be picked wherever it sits in the file,
            # and the same seed always picks the same documents.
            order = list(range(len(doc_lengths)))
            random.Random(SAMPLING_SEED).shuffle(order)

            # 1) HELD-OUT documents first (never trained on, never used for the tokenizer):
            #    whole documents long enough to hold a 512-word prompt + 128-word continuation,
            #    like the test examples. Used only for evaluation (Step 15).
            held_target = HELDOUT_WORDS.get(short, 0)
            heldout_ids, held_words = set(), 0
            for i in order:
                if held_words >= held_target:
                    break
                if doc_lengths[i] >= HELDOUT_MIN_DOC_WORDS:
                    heldout_ids.add(i)
                    held_words += min(doc_lengths[i], HELDOUT_MAX_WORDS_PER_DOC)

            # 2) TRAINING documents: continue down the same random order (skipping the
            #    held-out ones) until the word quota is met; the last one is truncated.
            chosen, planned = {}, 0
            for i in order:
                if planned >= quota:
                    break
                if i in heldout_ids:
                    continue
                keep = min(doc_lengths[i], quota - planned)
                if keep > 0:
                    chosen[i] = keep
                    planned += keep
            del order

            # PASS 2: write the chosen documents (in file order) to the corpus file, and the
            # held-out ones (one document per line) to their own file.
            source_word_count = 0
            remaining_docs, remaining_held = len(chosen), len(heldout_ids)
            held_f = None
            if heldout_ids:
                heldout_path = os.path.join(os.path.dirname(CORPUS_FILE), f"heldout_{short}.txt")
                held_f = open(heldout_path, "w", encoding="utf-8")
                HELDOUT_FILES[short] = heldout_path
            with open(local_path, "r", encoding="utf-8") as f:
                for i, record in enumerate(ijson.items(f, "item")):
                    keep = chosen.get(i)
                    if keep is None and i not in heldout_ids:
                        continue
                    doc_words = record[TEXT_FIELD].split()
                    if keep is not None:
                        out_f.write(" ".join(doc_words[:keep]) + " ")
                        source_word_count += min(keep, len(doc_words))
                        remaining_docs -= 1
                    else:
                        held_f.write(" ".join(doc_words[:HELDOUT_MAX_WORDS_PER_DOC]) + "\n")
                        remaining_held -= 1
                    if remaining_docs == 0 and remaining_held == 0:
                        break
            if held_f is not None:
                held_f.close()
            word_count += source_word_count
            print(f"  ...collected {source_word_count:,} words from {len(chosen):,} randomly chosen documents of "
                  f"{filename} (target {quota:,}); held out {len(heldout_ids):,} whole documents for evaluation")
    MAX_WORDS_FOR_TRAINING = word_count  # actual total = sum of quotas above

print(f"\nUsing {word_count} words for training (written to {CORPUS_FILE})")

Streaming through the training files, collecting words until we have enough...
  Phase_1_Pretraining/Train/train_acts.json: 18,826 documents, 119,638,348 words (counted in 10s)
  ...collected 60,000,000 words from 9,621 randomly chosen documents of Phase_1_Pretraining/Train/train_acts.json (target 60,000,000); held out 140 whole documents for evaluation


Phase_1_Pretraining/Train/train_high_cou(…): reconstructing file:   0%|          |  0.00B / 10.7GB            

Phase_1_Pretraining/Train/train_high_cou(…): downloading bytes:           |  0.00B            

  Phase_1_Pretraining/Train/train_high_courts.json: 2,536,738 documents, 1,594,919,242 words (counted in 125s)
  ...collected 60,000,000 words from 95,917 randomly chosen documents of Phase_1_Pretraining/Train/train_high_courts.json (target 60,000,000); held out 249 whole documents for evaluation


Phase_1_Pretraining/Train/train_supreme_(…): reconstructing file:   0%|          |  0.00B / 1.32GB            

Phase_1_Pretraining/Train/train_supreme_(…): downloading bytes:           |  0.00B            

  Phase_1_Pretraining/Train/train_supreme_court.json: 55,549 documents, 210,205,151 words (counted in 17s)
  ...collected 60,000,000 words from 15,543 randomly chosen documents of Phase_1_Pretraining/Train/train_supreme_court.json (target 60,000,000); held out 609 whole documents for evaluation


Phase_1_Pretraining/Train/train_tribunal(…): reconstructing file:   0%|          |  0.00B / 6.48GB            

Phase_1_Pretraining/Train/train_tribunal(…): downloading bytes:           |  0.00B            

  Phase_1_Pretraining/Train/train_tribunals.json: 869,132 documents, 975,917,446 words (counted in 83s)
  ...collected 60,000,000 words from 53,476 randomly chosen documents of Phase_1_Pretraining/Train/train_tribunals.json (target 60,000,000); held out 273 whole documents for evaluation

Using 240000000 words for training (written to /kaggle/working/bpe_training_corpus.txt)


## 5. Train a BPE (subword) tokenizer on the legal text -- our own implementation, from scratch

A word-level tokenizer with a small vocabulary throws away rare words as `<unk>`, which is bad for legal
text (case names, section numbers, Latin terms). **BPE (Byte-Pair Encoding)** instead learns to split rare
words into reusable pieces, so almost nothing becomes `<unk>`.

The tokenizer below is **not imported from any library**: it is the pure-Python byte-level BPE from
Assignment 1 (`BPETokenizer`), trained directly on this notebook's own corpus:

* words are split on whitespace and each word starts as its UTF-8 **bytes** plus an end-of-word marker
  `</w>`, so merges never cross word boundaries (and any text can be spelled out);
* training repeatedly merges the most frequent adjacent pair of symbols (ties: lexicographically smallest
  pair), until the vocabulary reaches `VOCAB_SIZE` or no pair occurs twice;
* encoding applies the learned merges to each word in the order they were learned; `decode` re-joins the
  words with single spaces. Special tokens: `<pad>`=0, `<unk>`=1, `<s>`=2, `</s>`=3.

**What is different from the Assignment-1 file, and why.** Only the *speed and memory* machinery, not the
algorithm: (1) word frequencies are counted by streaming the corpus file in blocks instead of holding the
text in memory; (2) the next merge is picked with a max-heap instead of scanning every pair each time
(scanning ~10^5-10^6 pairs for each of ~40,000 merges would take days on an 160M-word corpus); (3) each
word's token ids are cached, so the 160M-word corpus is encoded with one dictionary lookup per word. The
heap version selects exactly the same merge at every step as the Assignment-1 scan (same tie-break rule);
this was checked against the original implementation on random corpora, comparing the merge lists,
vocabulary, encodings and decodings.

Because it is plain deterministic Python, the same corpus and `VOCAB_SIZE` always give the same
vocabulary, which is what lets a saved checkpoint be resumed later. Training this in pure Python on 160M
words takes a while (progress is printed; not yet timed on the real corpus). `BPE_MIN_WORD_FREQ = 1` uses
every word type exactly like Assignment 1; raising it to 2 skips words that occur only once when *learning*
merges (faster and lighter, slightly different merges); encoding always handles every word.

**Saving the tokenizer.** Training this tokenizer is the slowest pre-training step, so the trained merges are
saved (`a1bpe_tokenizer_<mode>_v<vocab>_<words>w_seed<seed>_minf<n>.json`, in `/kaggle/working`) and pushed to the
same Hugging Face repo as the checkpoints. The file name records everything the tokenizer depends on (corpus size,
sampling seed, vocabulary size, `BPE_MIN_WORD_FREQ`). On a re-run -- or from the other Kaggle account -- the cell finds
the file and loads it instead of training; change any of those settings and it trains a new one.

In [9]:
import array, heapq, time
from collections import Counter, defaultdict

BPE_MIN_WORD_FREQ = 1      # 1 = learn merges from every word type (exactly the Assignment-1 algorithm)
_END = "</w>"


class BPETokenizer:
    """Byte-level BPE (Assignment 1). Initial symbols are the UTF-8 bytes of each word, held as chr(b), plus
    '</w>' marking the end of a word so merges never cross word boundaries. Ties in pair frequency break
    lexicographically. Text is NOT lowercased."""

    def __init__(self, vocab_size):
        self.vocab_size = vocab_size
        self.merges = []
        self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
        self.stoi = {}
        self._ranks = {}
        self._id_cache = {}

    # ---- helpers ----
    def _word_symbols(self, word):
        syms = [chr(b) for b in word.encode("utf-8")]
        syms.append(_END)
        return syms

    def _merge(self, symbols, pair):
        res, i = [], 0
        a, b = pair
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                res.append(a + b)
                i += 2
            else:
                res.append(symbols[i])
                i += 1
        return res

    # ---- training ----
    def fit_file(self, path, block_chars=50_000_000, min_word_freq=1):
        """Count word frequencies by streaming the corpus file in blocks, then learn the merges."""
        t0 = time.time()
        word_freqs = Counter()
        leftover = ""
        with open(path, "r", encoding="utf-8") as f:
            while True:
                block = f.read(block_chars)
                if not block:
                    break
                block = leftover + block
                parts = block.split()
                if parts and not block[-1].isspace():
                    leftover = parts.pop()          # the last word may be cut in half: finish it with the next block
                else:
                    leftover = ""
                word_freqs.update(parts)
        if leftover:
            word_freqs[leftover] += 1
        print(f"  counted {sum(word_freqs.values()):,} words ({len(word_freqs):,} distinct) in {time.time() - t0:.0f}s", flush=True)
        return self.fit_word_freqs(word_freqs, min_word_freq)

    def fit_word_freqs(self, word_freqs, min_word_freq=1):
        t0 = time.time()
        target = self.vocab_size
        words = [w for w, c in word_freqs.items() if c >= min_word_freq]
        freqs = [word_freqs[w] for w in words]
        splits = [self._word_symbols(w) for w in words]
        del word_freqs

        base = set()
        for syms in splits:
            base.update(syms)
        self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
        self.merges = []
        self.itos.extend(sorted(base))
        print(f"  base byte alphabet: {len(base)} symbols; learning merges from {len(words):,} word types", flush=True)

        # pair_counts: how often each adjacent pair occurs (weighted by word frequency).
        # pair_words: which word types contain each pair, so a merge only revisits those words.
        pair_counts = {}
        pair_words = defaultdict(set)
        for i, syms in enumerate(splits):
            f = freqs[i]
            for j in range(len(syms) - 1):
                p = (syms[j], syms[j + 1])
                pair_counts[p] = pair_counts.get(p, 0) + f
                pair_words[p].add(i)

        # Max-heap of (-count, pair): pops the most frequent pair, ties -> lexicographically smallest pair
        # (identical to Assignment 1's min over (-count, pair)). Entries go stale when a count changes; a
        # fresh entry is pushed for every changed pair and stale ones are skipped when they reach the top.
        heap = [(-c, p) for p, c in pair_counts.items()]
        heapq.heapify(heap)

        while len(self.itos) < target:
            best, best_f = None, 0
            while heap:
                neg_c, p = heap[0]
                if pair_counts.get(p, 0) == -neg_c:
                    best, best_f = p, -neg_c
                    heapq.heappop(heap)
                    break
                heapq.heappop(heap)                  # stale entry
            if best is None:
                break
            if best_f < 2:
                print("  stopping: no pair occurs more than once", flush=True)
                break

            self.merges.append([best[0], best[1]])
            self.itos.append(best[0] + best[1])

            changed = set()
            for i in list(pair_words[best]):
                syms, f = splits[i], freqs[i]
                for j in range(len(syms) - 1):
                    p = (syms[j], syms[j + 1])
                    c = pair_counts.get(p, 0) - f
                    if c <= 0:
                        pair_counts.pop(p, None)
                    else:
                        pair_counts[p] = c
                    pair_words[p].discard(i)
                    changed.add(p)
                new_syms = self._merge(syms, best)
                splits[i] = new_syms
                for j in range(len(new_syms) - 1):
                    p = (new_syms[j], new_syms[j + 1])
                    pair_counts[p] = pair_counts.get(p, 0) + f
                    pair_words[p].add(i)
                    changed.add(p)
            for p in changed:
                c = pair_counts.get(p)
                if c:
                    heapq.heappush(heap, (-c, p))
            del pair_words[best]

            if len(self.merges) % 1000 == 0:
                print(f"  merges {len(self.merges):,} | vocab {len(self.itos):,} | {time.time() - t0:.0f}s", flush=True)

        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self._ranks = {(a, b): r for r, (a, b) in enumerate(self.merges)}
        self._id_cache = {}
        self.vocab_size = len(self.itos)      # the size the embedding table must actually have
        return self

    # ---- save / load (so the tokenizer is trained once and then reused) ----
    def save(self, path, meta=None):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"vocab_size": self.vocab_size, "merges": self.merges, "itos": self.itos,
                       "meta": meta or {}}, f)

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        tok = cls(data["vocab_size"])
        tok.merges = [list(m) for m in data["merges"]]
        tok.itos = list(data["itos"])
        tok.stoi = {t: i for i, t in enumerate(tok.itos)}
        tok._ranks = {(a, b): r for r, (a, b) in enumerate(tok.merges)}
        tok._id_cache = {}
        tok.vocab_size = len(tok.itos)
        return tok

    # ---- inference ----
    def _tokenize_word(self, word):
        syms = self._word_symbols(word)
        while len(syms) > 1:
            best_rank, best_j = None, -1
            for j in range(len(syms) - 1):
                r = self._ranks.get((syms[j], syms[j + 1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best_rank, best_j = r, j
            if best_rank is None:
                break
            syms = self._merge(syms, (syms[best_j], syms[best_j + 1]))
        return syms

    def _word_ids(self, word):
        ids = self._id_cache.get(word)
        if ids is None:
            unk = self.stoi["<unk>"]
            ids = [self.stoi.get(t, unk) for t in self._tokenize_word(word)]     # symbols never seen in training -> <unk>
            self._id_cache[word] = ids
        return ids

    def tokenize(self, text):
        out = []
        for word in str(text).split():
            out.extend(self._tokenize_word(word))
        return out

    def get_vocab_size(self):
        return len(self.itos)

    def encode(self, text):
        out = []
        for word in str(text).split():
            out.extend(self._word_ids(word))
        return out

    def encode_file(self, path, block_chars=5_000_000):
        """Encode a whole text file into a packed array of ids, block by block (bounded memory)."""
        out = array.array("i")
        leftover = ""
        with open(path, "r", encoding="utf-8") as f:
            while True:
                block = f.read(block_chars)
                if not block:
                    break
                block = leftover + block
                parts = block.split()
                if parts and not block[-1].isspace():
                    leftover = parts.pop()
                else:
                    leftover = ""
                cache = self._id_cache
                for word in parts:
                    ids = cache.get(word)
                    if ids is None:
                        ids = self._word_ids(word)
                    out.extend(ids)
        if leftover:
            out.extend(self._word_ids(leftover))
        return out

    def decode(self, ids):
        """ids -> string. Words are re-joined with single spaces; <unk> decodes to '?'."""
        words, cur = [], bytearray()
        for i in ids:
            i = int(i)
            if not 0 <= i < len(self.itos):
                continue
            tok = self.itos[i]
            if tok in ("<pad>", "<s>", "</s>"):
                continue
            if tok == "<unk>":
                cur.extend(b"?")
                continue
            ends_word = tok.endswith(_END)
            if ends_word:
                tok = tok[: -len(_END)]
            cur.extend(ord(c) & 0xFF for c in tok)
            if ends_word:
                words.append(bytes(cur).decode("utf-8", errors="replace"))
                cur = bytearray()
        if cur:
            words.append(bytes(cur).decode("utf-8", errors="replace"))
        return " ".join(words)


# The tokenizer depends only on the training corpus, VOCAB_SIZE and BPE_MIN_WORD_FREQ, so its file name records
# exactly those (the corpus is fixed by the sampling seed, the quotas and the word count). If a file with this name is
# on disk or on the Hub it is loaded (skipping the slow pure-Python training); otherwise it is trained, saved and pushed.
TOKENIZER_PATH = (f"/kaggle/working/a1bpe_tokenizer_{TOKENIZER_RUN_TAG}_{mode_tag}_v{VOCAB_SIZE}_{word_count if QUICK_TEST else TOKENIZER_WORDS}w"
                  f"_seed{SAMPLING_SEED}_minf{BPE_MIN_WORD_FREQ}.json")
t_start = time.time()
if pull_from_hub(TOKENIZER_PATH):
    print(f"Loading the saved tokenizer {os.path.basename(TOKENIZER_PATH)} (skipping training)...")
    bpe_tokenizer = BPETokenizer.load(TOKENIZER_PATH)
    tokenizer_source = "loaded"
else:
    if REQUIRE_SAVED_TOKENIZER and not QUICK_TEST:
        raise RuntimeError(f"The saved 160M-word tokenizer {os.path.basename(TOKENIZER_PATH)} was not found locally or in the bucket {HF_BUCKET}. "
                           "This run is meant to reuse the tokenizer of the other 160M-word runs (same corpus, same vocabulary). Check the HF token / bucket, "
                           "or set REQUIRE_SAVED_TOKENIZER = False in Settings to train a new one (slow, and token ids will differ from the other runs).")
    print("Training the BPE tokenizer (our own implementation, pure Python)...")
    bpe_tokenizer = BPETokenizer(vocab_size=VOCAB_SIZE).fit_file(CORPUS_FILE, min_word_freq=BPE_MIN_WORD_FREQ)
    bpe_tokenizer.save(TOKENIZER_PATH, meta={"word_count": word_count, "sampling_seed": SAMPLING_SEED,
                                             "min_word_freq": BPE_MIN_WORD_FREQ, "requested_vocab": VOCAB_SIZE})
    try:
        push_checkpoint_to_hub(TOKENIZER_PATH)
    except Exception as e:
        print(f"(could not push the tokenizer to the Hub: {type(e).__name__} -- it is still saved locally)")
    tokenizer_source = "trained"
print(f"Tokenizer {tokenizer_source} in {time.time() - t_start:.0f}s")
log_event("tokenizer", source=tokenizer_source, seconds=round(time.time() - t_start), vocab_size=bpe_tokenizer.get_vocab_size(),
          merges=len(bpe_tokenizer.merges), words=word_count, file=os.path.basename(TOKENIZER_PATH))


def encode(text):
    """Turn a string into a list of BPE token IDs."""
    return bpe_tokenizer.encode(text)


def decode(ids):
    """Turn a list of BPE token IDs back into a string."""
    return bpe_tokenizer.decode(ids)


print("Vocabulary size:", bpe_tokenizer.get_vocab_size())
if bpe_tokenizer.get_vocab_size() < VOCAB_SIZE:
    print(f"NOTE: fewer than VOCAB_SIZE={VOCAB_SIZE} merges were possible; the model uses the real size above.")

# Encode the whole training corpus (streamed from disk, one id-cache lookup per word). The array is named
# all_word_ids for consistency with the other notebooks; it holds BPE-token ids, not one id per word.
t_start = time.time()
all_word_ids = bpe_tokenizer.encode_file(CORPUS_FILE)
print(f"{word_count} words became {len(all_word_ids)} BPE tokens "
      f"({len(all_word_ids) / word_count:.2f} tokens per word) in {time.time() - t_start:.0f}s")
log_event("encoded_corpus", words=word_count, tokens=len(all_word_ids), tokens_per_word=round(len(all_word_ids) / word_count, 3),
          seconds=round(time.time() - t_start))

(…)full_v40000_160000000w_seed42_minf1.json:   0%|          | 0.00/1.37M [00:00<?, ?B/s]

(found a1bpe_tokenizer_160Mw_1ep_full_v40000_160000000w_seed42_minf1.json only in the old repo kavincg23/cs781-checkpoints; new saves go to the bucket)
Pulled a1bpe_tokenizer_160Mw_1ep_full_v40000_160000000w_seed42_minf1.json from the Hub
Loading the saved tokenizer a1bpe_tokenizer_160Mw_1ep_full_v40000_160000000w_seed42_minf1.json (skipping training)...
Tokenizer loaded in 1s
Vocabulary size: 40000


/tmp/ipykernel_23/2161731718.py:103: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


240000000 words became 276362790 BPE tokens (1.15 tokens per word) in 257s


## 6. Chop the training text into fixed-size chunks

We cut the long stream of word-IDs into pieces of length `CONTEXT_LEN`
(640 words). Each piece is one training example. A model learns to predict
the next word at every position in the piece.

*Simplification:* this glues all documents together into one long stream
before cutting, so a few chunks will mix the end of one case with the start
of another. That's fine for a first version — separating documents cleanly
is one of the "task-specific preprocessing" experiments you could try later.

In [10]:
# Build the chunk tensor straight from the packed int array (no per-token Python objects: with ~200M tokens a list of lists
# would need many GB of RAM). Same chunks as before: starts 0, CONTEXT_LEN, 2*CONTEXT_LEN, ... while start < len - CONTEXT_LEN.
num_blocks = max(0, -(-(len(all_word_ids) - CONTEXT_LEN) // CONTEXT_LEN))
_flat = torch.frombuffer(all_word_ids, dtype=torch.int32) if isinstance(all_word_ids, array.array) else torch.tensor(all_word_ids, dtype=torch.int32)
data = _flat[:num_blocks * CONTEXT_LEN].view(num_blocks, CONTEXT_LEN)   # stays int32 (half the RAM); each batch is converted with .long()
del _flat
print(f"Number of training chunks: {num_blocks}")

# Shuffle the chunks (fixed seed, so this is reproducible) BEFORE the
# train/val split below. The corpus is source-blocked -- all of one
# source's words, then the next -- so without this, the validation slice
# (a prefix of `data`) would be drawn entirely from whichever source comes
# first (train_acts.json), not proportionally from all 4 sources. Training
# itself doesn't need this shuffle (the training loop below already
# reshuffles chunk order every epoch via torch.randperm), but the
# validation split is fixed once up front, so it needs its own shuffle.
val_split_seed = torch.Generator().manual_seed(42)
shuffled_chunk_order = torch.randperm(len(data), generator=val_split_seed)
data = data[shuffled_chunk_order]

# hold out a slice for validation / local testing -- a non-overlapping
# slice of (now-shuffled) chunks, so no word is ever shared between
# train_data and val_data
num_val = max(1, int(0.01 * len(data)))
val_data = data[:num_val]
train_data = data[num_val:]
print(f"Training chunks: {len(train_data)}, validation chunks: {len(val_data)}")

Number of training chunks: 134942
Training chunks: 133593, validation chunks: 1349


## 7. Building blocks for the research knobs

Before defining the model itself, here are the pieces `NORM_TYPE`,
`NORM_POSITION` and `OPTIMIZER_TYPE` (and `ATTENTION_TYPE`, for this
architecture) plug into. Every architecture in all 4 notebooks uses these
same building blocks, so results are comparable across notebooks.

In [11]:
class RMSNorm(nn.Module):
    """Like LayerNorm, but simpler: it only rescales by the root-mean-square
    of the values, it does not re-center the mean. Cheaper, and used in
    models like LLaMA."""

    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).sqrt()
        return x / rms * self.weight


def make_norm(dim, norm_type):
    """norm_type is "layernorm", "rmsnorm" or "deepnorm".
    deepnorm reuses a plain LayerNorm -- what makes deepnorm special isn't
    the norm formula, it's the residual scaling (alpha) and weight init
    (beta) we apply around it below."""
    if norm_type == "rmsnorm":
        return RMSNorm(dim)
    else:  # "layernorm" or "deepnorm"
        return nn.LayerNorm(dim)


def deepnorm_alpha(n_layers):
    """Residual scaling factor from the DeepNet paper: alpha = (2N)^0.25."""
    return (2.0 * n_layers) ** 0.25


def deepnorm_init(output_projections, n_layers):
    """Scales the init of a sub-layer's OUTPUT projection(s) (e.g. attention
    out_proj, the feed-forward's second linear) by beta = (8N)^-0.25, so a
    deep post-norm stack stays stable to train. Only used when
    NORM_TYPE == "deepnorm"."""
    beta = (8.0 * n_layers) ** -0.25
    for module in output_projections:
        nn.init.xavier_normal_(module.weight, gain=beta)
        if module.bias is not None:
            nn.init.zeros_(module.bias)

### Optimizer with `OPTIMIZER_TYPE` support (AdamW / Muon)

**Muon** (Keller Jordan) keeps a momentum buffer like normal SGD, but
before applying the update it "orthogonalizes" it (a fast Newton-Schulz
iteration pushes every singular value of the update matrix towards 1).
This only makes sense for 2D+ weight matrices — biases, embeddings and
norm weights still use plain AdamW underneath. This is the standard Muon
recipe (and why `build_optimizer` below actually returns two optimizers
wrapped so the training loop can use them like one).

In [12]:
# Fused AdamW: PyTorch runs the whole AdamW update in a few fused GPU kernels instead of many small ones.
# Same math, slightly faster. It needs the parameters on a CUDA GPU, so it is off on CPU.
USE_FUSED_ADAMW = torch.cuda.is_available()


def zeropower_via_newton_schulz(update_matrix, steps=5, eps=1e-7):
    """Orthogonalize a matrix (push its singular values towards 1) with a
    fast iterative approximation. This is the core trick behind Muon."""
    a, b, c = 3.4445, -4.7750, 2.0315
    x = update_matrix.float()
    flipped = x.size(-2) > x.size(-1)
    if flipped:
        x = x.transpose(-2, -1)
    x = x / (x.norm(dim=(-2, -1), keepdim=True) + eps)
    for _ in range(steps):
        a_term = x @ x.transpose(-2, -1)
        b_term = b * a_term + c * a_term @ a_term
        x = a * x + b_term @ x
    if flipped:
        x = x.transpose(-2, -1)
    return x


class Muon(torch.optim.Optimizer):
    """Simplified Muon optimizer. Only meant to be given 2D+ weight matrices."""

    def __init__(self, params, lr=0.02, momentum=0.95, ns_steps=5):
        super().__init__(params, dict(lr=lr, momentum=momentum, ns_steps=ns_steps))

    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None:
                    continue
                state = self.state[p]
                if "momentum_buffer" not in state:
                    state["momentum_buffer"] = torch.zeros_like(p.grad)
                buf = state["momentum_buffer"]
                buf.mul_(group["momentum"]).add_(p.grad)
                update = zeropower_via_newton_schulz(buf, steps=group["ns_steps"])
                scale = max(1.0, p.grad.size(-2) / p.grad.size(-1)) ** 0.5
                p.add_(update.reshape(p.shape).to(p.dtype), alpha=-group["lr"] * scale)


class CombinedOptimizer:
    """Wraps two optimizers so the training loop can call .zero_grad() /
    .step() on it just like a single normal optimizer."""

    def __init__(self, optimizers):
        self.optimizers = optimizers

    def zero_grad(self):
        for opt in self.optimizers:
            opt.zero_grad()

    def step(self):
        for opt in self.optimizers:
            opt.step()

    def state_dict(self):
        return [opt.state_dict() for opt in self.optimizers]

    def load_state_dict(self, state_dicts):
        for opt, state in zip(self.optimizers, state_dicts):
            opt.load_state_dict(state)


def build_optimizer(model, optimizer_type, lr):
    """optimizer_type is "adamw" or "muon"."""
    if optimizer_type == "adamw":
        return torch.optim.AdamW(model.parameters(), lr=lr, fused=USE_FUSED_ADAMW)

    if optimizer_type == "muon":
        muon_params, adamw_params = [], []
        for name, p in model.named_parameters():
            # muon only works well on 2D+ "matrix" weights; everything else
            # (biases, norm weights, embedding tables) stays on adamw
            if p.ndim >= 2 and "emb" not in name.lower():
                muon_params.append(p)
            else:
                adamw_params.append(p)
        muon_opt = Muon(muon_params, lr=lr * 30)   # muon typically wants a bigger lr than adamw
        adamw_opt = torch.optim.AdamW(adamw_params, lr=lr, fused=USE_FUSED_ADAMW)
        return CombinedOptimizer([muon_opt, adamw_opt])

    raise ValueError(f"unknown optimizer_type {optimizer_type}")

## 8. The CALM architecture

**Stage 1 — the autoencoder (`ChunkAutoencoder`).** A small variational autoencoder with no attention. For one chunk of K = 4 token ids:

| step | shape |
|---|---|
| embed each token (table V × `AE_D`) | 4 × 512 |
| FFN on each token | 4 × 512 |
| flatten, Linear → one vector | 2048 → 512 |
| FFN, LayerNorm, Linear → (mean, log-variance) | 2 × 128 |
| **decoder:** sample z, Linear, FFN, Linear, split, FFN per position | 128 → 512 → 2048 → 4 × 512 |
| output = decoder vectors × (embedding table)ᵀ (**tied**) | 4 × V logits |

Loss = reconstruction cross-entropy (sum over the 4 tokens) + 0.001 × KL with a per-dimension free-bits clip of 0.5. Two robustness
tricks from the paper: 15 % of the *input* tokens are blanked and the latent goes through 15 % dropout, so the decoder tolerates imperfect
vectors (the language model's guesses will be imperfect). Stage 1 **stops early** once held-out per-token accuracy (decoding a *sampled*
latent) reaches `AE_TARGET_ACC`.

**Stage 2 — the language model.** For a window of 2048 tokens = 512 chunks:
* **Input block (`ChunkInput`)** — the 4 token ids of a chunk → 4 vectors from the backbone's *own* table (V × 768) → flatten (3072) →
  Linear → GELU → Linear → **one 768-vector**. This is the only place the 4 tokens of a chunk interact (optionally also through `LocalMix`).
* **Backbone (`CALMBackbone`)** — 12 pre-norm layers, width 768, 12 query heads / **6 key-value heads** (GQA, 2 queries per KV head), head
  dim 64, RoPE over the *chunk index*, FFN width `D_FF`. Position *i* sees chunks 0…i and its output hᵢ (768) is used to guess chunk i+1.
* **Energy head (`EnergyHead`)** — hᵢ plus random noise → a guess ẑ of the next chunk's 128-number latent. It is called N = 8 times with
  fresh noise to get 8 different guesses.
* **Loss (`energy_score_loss`)** — with M = 100 samples zₘ from the frozen autoencoder's posterior for the *real* next chunk:
  `2 · mean‖zₘ − ẑₙ‖  −  mean over pairs n≠k of ‖ẑₙ − ẑₖ‖`. The first term pulls every guess towards the real latent; the second pushes the
  guesses apart so the head does not collapse to a single answer. It is a strictly proper scoring rule: its minimum is the true distribution.

**Generation.** Prefill the prompt (a whole number of chunks; 0–3 of the oldest tokens are dropped), then repeat: backbone → hᵢ → draw
`GEN_CANDIDATES` guesses → decode them to tokens with the autoencoder → keep the chunk most guesses agree on (`GEN_SELECT = "vote"`) →
feed its 4 tokens back in. Stop after 128 words. A 128-word continuation is only ~50 backbone steps.

In [13]:
import math

ROPE_BASE = 10000.0     # RoPE frequency base: theta_i = ROPE_BASE ** (-2i / head_dim); positions are CHUNK indices here
USE_NATIVE_GQA = False  # False: expand K/V with repeat_interleave before SDPA (the baseline's default when its GPU benchmark did not favour enable_gqa)


def build_rope_cache(max_seq_len, head_dim, base=10000.0):
    """cos/sin tables for RoPE, each (max_seq_len, head_dim). Split-half
    convention: channel i is paired with channel i + head_dim/2, so the angle
    table is cat([freqs, freqs]) (NOT repeat_interleave, which pairs neighbours
    instead). Same construction as the RoPE written for Assignment 3."""
    assert head_dim % 2 == 0, "RoPE requires an even head dimension"
    inv_freq = 1.0 / (base ** (2 * torch.arange(0, head_dim // 2).float() / head_dim))
    positions = torch.arange(0, max_seq_len).float()
    freqs = torch.outer(positions, inv_freq)              # (max_seq_len, head_dim // 2)
    angles = torch.cat([freqs, freqs], dim=-1)            # (max_seq_len, head_dim)
    return angles.cos(), angles.sin()


def rotate_half(x):
    """Split the last dim in half and return (-x2, x1): a 90-degree rotation step."""
    x1, x2 = torch.chunk(x, 2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(x, cos, sin):
    """x_rotated = x * cos + rotate_half(x) * sin. x: (..., head_dim); cos/sin
    already broadcastable against x (the model gathers them per position). The
    arithmetic is done in fp32 (even under fp16 autocast) and cast back, so the
    rotation stays accurate and q/k keep the same dtype as v for SDPA."""
    xf = x.float()
    return (xf * cos + rotate_half(xf) * sin).to(x.dtype)


class KVCache:
    """Pre-allocated key/value storage for EVERY layer, written in place (the
    'static cache' idea from gpt-fast). Shapes are (batch, n_kv_heads, max_len,
    head_dim) -- n_kv_heads, not n_heads, which is where GQA's 4x saving
    lives. `length` is how many positions are currently filled (the same for
    every row in the batch: shorter prompts are left-padded)."""

    def __init__(self, n_layers, batch_size, n_kv_heads, max_len, head_dim, device, dtype):
        shape = (batch_size, n_kv_heads, max_len, head_dim)
        self.k = [torch.zeros(shape, device=device, dtype=dtype) for _ in range(n_layers)]
        self.v = [torch.zeros(shape, device=device, dtype=dtype) for _ in range(n_layers)]
        self.length = 0
        self.max_len = max_len

    def reset(self):
        self.length = 0

    def num_bytes(self):
        return sum(t.numel() * t.element_size() for t in self.k + self.v)


class CachedGQAttention(nn.Module):
    """Grouped-query self-attention (causal) with two entry points:
      forward()      -- the parallel form used for training and for PREFILL.
                        Optionally also stores this call's keys/values in a cache.
      decode_step()  -- ONE new token, reading/writing the cache.
    Parameter names (q_proj/k_proj/v_proj/out_proj) match the original
    MultiHeadAttention with ATTENTION_TYPE="gqa", so checkpoints are interchangeable."""

    def __init__(self, d_model, n_heads, n_kv_heads):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads
        self.group_size = n_heads // n_kv_heads

        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim)
        self.out_proj = nn.Linear(n_heads * self.head_dim, d_model)

    def forward(self, x, key_padding_mask=None, k_cache=None, v_cache=None, rope=None):
        """x: (batch, seq, d_model). key_padding_mask (optional): (batch, seq)
        bool, True = real (non-pad) position; leave None when nothing is
        padded so the fused kernels can run mask-free. If k_cache/v_cache are
        given, this call's (un-expanded) keys/values are written to
        positions [0:seq] of them -- that is the PREFILL step. rope (optional):
        (cos, sin), each (batch or 1, 1, seq, head_dim), gathered from the
        model's RoPE tables -- None when POSITIONAL_ENCODING == "learned"."""
        batch_size, seq_len, _ = x.shape
        q = self.q_proj(x).view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(batch_size, seq_len, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(batch_size, seq_len, self.n_kv_heads, self.head_dim).transpose(1, 2)

        if rope is not None:
            # rotate BEFORE caching: the cache stores already-rotated keys (each
            # key is rotated by its own position exactly once, when it is written)
            q = apply_rope(q, *rope)
            k = apply_rope(k, *rope)

        if k_cache is not None:
            k_cache[:, :, :seq_len] = k.to(k_cache.dtype)
            v_cache[:, :, :seq_len] = v.to(v_cache.dtype)

        attn_mask = None
        if key_padding_mask is not None:
            positions = torch.arange(seq_len, device=x.device)
            causal = positions.unsqueeze(1) >= positions.unsqueeze(0)               # (seq, seq)
            allowed = causal.unsqueeze(0) & key_padding_mask.unsqueeze(1)           # (batch, seq, seq)
            # a query may always see itself -- guarantees no row is fully
            # masked (which would produce NaNs for pad-position queries)
            allowed = allowed | torch.eye(seq_len, dtype=torch.bool, device=x.device).unsqueeze(0)
            attn_mask = allowed.unsqueeze(1)                                        # (batch, 1, seq, seq)

        use_causal_flag = attn_mask is None
        if USE_NATIVE_GQA and self.group_size > 1:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask,
                                                 is_causal=use_causal_flag, enable_gqa=True)
        else:
            if self.group_size > 1:
                k = k.repeat_interleave(self.group_size, dim=1)
                v = v.repeat_interleave(self.group_size, dim=1)
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask,
                                                 is_causal=use_causal_flag)

        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, self.n_heads * self.head_dim)
        return self.out_proj(out)

    def decode_step(self, x_t, k_cache, v_cache, cache_len, key_mask=None, rope=None):
        """x_t: (batch, d_model) -- ONE new position per row. Writes its K/V at
        index cache_len and attends over cache positions 0..cache_len
        (inclusive). key_mask (optional): (batch, cache_len + 1) bool, True =
        real key; None when the batch has no padding. rope (optional): (cos,
        sin), each (batch, head_dim) for this step's position."""
        batch_size = x_t.shape[0]

        # Fold the `group_size` query heads that share one K/V head into the
        # query-LENGTH axis: (batch, n_kv_heads, group_size, head_dim). The
        # cache is then used as-is (never expanded), and because the new token
        # is the last position no causal mask is needed.
        q = self.q_proj(x_t).view(batch_size, self.n_kv_heads, self.group_size, self.head_dim)
        k_new = self.k_proj(x_t).view(batch_size, self.n_kv_heads, self.head_dim)
        v_new = self.v_proj(x_t).view(batch_size, self.n_kv_heads, self.head_dim)

        if rope is not None:
            cos, sin = rope
            q = apply_rope(q, cos.view(batch_size, 1, 1, -1), sin.view(batch_size, 1, 1, -1))
            k_new = apply_rope(k_new, cos.view(batch_size, 1, -1), sin.view(batch_size, 1, -1))

        k_cache[:, :, cache_len] = k_new.to(k_cache.dtype)
        v_cache[:, :, cache_len] = v_new.to(v_cache.dtype)
        k = k_cache[:, :, :cache_len + 1]
        v = v_cache[:, :, :cache_len + 1]
        if q.dtype != k.dtype:
            q = q.to(k.dtype)

        attn_mask = None if key_mask is None else key_mask.view(batch_size, 1, 1, -1)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask)   # (batch, n_kv, group, head_dim)
        out = out.reshape(batch_size, self.n_heads * self.head_dim).to(x_t.dtype)
        return self.out_proj(out)


# =====================================================================================================
# Everything below is NEW for CALM (the RoPE / KV-cache / grouped-query attention code above is the
# baseline's, unchanged). Three pieces:
#   1. ChunkAutoencoder  -- Stage 1: K tokens <-> one LATENT_DIM-number vector (a small VAE, no attention)
#   2. CALMBackbone      -- Stage 2: a 12-layer GQA transformer over ONE position per chunk of K tokens
#   3. EnergyHead        -- Stage 2: turns the backbone's output into random guesses of the next latent
# plus the energy-score loss that trains the head WITHOUT a likelihood (no softmax over the vocabulary).
# =====================================================================================================

PAD_ID = 0   # the BPE tokenizer's special_tokens list put "<pad>" first, so its id is 0 (never appears in real text)


class FFN(nn.Module):
    """Two linear layers with a GELU in between: d -> ff -> d."""

    def __init__(self, d, ff):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Linear(ff, d))

    def forward(self, x):
        return self.net(x)


class ResFFN(nn.Module):
    """x + FFN(norm(x)) -- the pre-norm residual feed-forward block used inside the autoencoder and the head."""

    def __init__(self, d, ff):
        super().__init__()
        self.norm = make_norm(d, NORM_TYPE)
        self.ff = FFN(d, ff)

    def forward(self, x):
        return x + self.ff(self.norm(x))


# ----------------------------------------------------------------------------------------------------
# Stage 1: the autoencoder (chunk of K tokens  <->  one latent vector of LATENT_DIM numbers)
# ----------------------------------------------------------------------------------------------------
class ChunkAutoencoder(nn.Module):
    """A small variational autoencoder with NO attention (the K tokens of a chunk are mixed by a
    flatten-and-project step, like a 'token mixer'). Shapes for one chunk, with K=4, d=AE_D, l=LATENT_DIM:

        encode:  K token ids -> embed (K,d) -> FFN per token -> flatten (K*d) -> Linear -> d -> FFN -> LN
                 -> Linear -> (mu, logvar), each l numbers
        decode:  z (l) -> Linear -> d -> FFN -> Linear -> (K*d) -> split into K vectors of d -> FFN per
                 position -> LN -> multiply by the (tied) embedding table -> K sets of vocabulary logits

    The output layer is the transpose of the input embedding table (weight tying), so the autoencoder has
    only one V x d table. Everything here is fp32."""

    def __init__(self, vocab_size, k, d, ff, latent_dim):
        super().__init__()
        self.k, self.d, self.latent_dim, self.vocab_size = k, d, latent_dim, vocab_size
        self.emb = nn.Embedding(vocab_size, d)
        nn.init.normal_(self.emb.weight, std=d ** -0.5)   # small, so the tied output logits start at a sane scale
        self.enc_tok = ResFFN(d, ff)
        self.enc_merge = nn.Linear(k * d, d)
        self.enc_mix = ResFFN(d, ff)
        self.enc_norm = make_norm(d, NORM_TYPE)
        self.enc_out = nn.Linear(d, 2 * latent_dim)
        self.dec_in = nn.Linear(latent_dim, d)
        self.dec_mix = ResFFN(d, ff)
        self.dec_split = nn.Linear(d, k * d)
        self.dec_tok = ResFFN(d, ff)
        self.dec_norm = make_norm(d, NORM_TYPE)

    def encode(self, tokens):
        """tokens: (..., K) long -> (mu, logvar), each (..., latent_dim)."""
        h = self.enc_tok(self.emb(tokens))                    # (..., K, d)
        h = self.enc_merge(h.flatten(-2))                     # (..., d)   token order is kept by the flatten
        h = self.enc_mix(h)
        mu, logvar = self.enc_out(self.enc_norm(h)).chunk(2, dim=-1)
        return mu, logvar.clamp(-10.0, 4.0)

    def decode_logits(self, z):
        """z: (..., latent_dim) -> logits (..., K, vocab_size)."""
        h = self.dec_mix(self.dec_in(z))                      # (..., d)
        h = self.dec_split(h).unflatten(-1, (self.k, self.d)) # (..., K, d)
        h = self.dec_norm(self.dec_tok(h))
        return h @ self.emb.weight.t()                        # tied output layer

    def forward(self, tokens):
        """TRAINING forward. tokens: (N, K) long. Returns (loss, stats). Robustness tricks from the paper:
        random input tokens are blanked (AE_TOKEN_MASK) and the latent goes through dropout (AE_LATENT_DROPOUT),
        so the decoder learns to cope with imperfect latents (the language model's guesses will be imperfect)."""
        inp = tokens
        if self.training and AE_TOKEN_MASK > 0:
            inp = tokens.masked_fill(torch.rand(tokens.shape, device=tokens.device) < AE_TOKEN_MASK, PAD_ID)
        mu, logvar = self.encode(inp)
        z = mu + (0.5 * logvar).exp() * torch.randn_like(mu)   # sample from the posterior
        z = F.dropout(z, AE_LATENT_DROPOUT, self.training)
        logits = self.decode_logits(z)
        rec = F.cross_entropy(logits.reshape(-1, self.vocab_size), tokens.reshape(-1), reduction="sum") / tokens.shape[0]
        kl_dim = 0.5 * (mu.pow(2) + logvar.exp() - logvar - 1.0)   # (N, l) KL of each dimension to N(0, 1)
        kl = kl_dim.mean(dim=0).clamp(min=AE_KL_CLIP).sum()         # per-dimension "free bits": no penalty below AE_KL_CLIP
        loss = rec + AE_KL_WEIGHT * kl
        with torch.no_grad():
            acc = (logits.argmax(-1) == tokens).float().mean()
        return loss, {"rec": rec.detach(), "kl": kl_dim.mean().detach(), "tok_acc": acc}

    @torch.no_grad()
    def reconstruction_accuracy(self, tokens, sample_latent=True):
        """Eval-mode accuracy on (N, K) token chunks: (per-token accuracy, whole-chunk accuracy).
        sample_latent=True decodes mu + sigma*noise (what a slightly-off language-model guess looks like);
        False decodes the posterior mean."""
        was_training = self.training
        self.eval()
        mu, logvar = self.encode(tokens)
        z = mu + (0.5 * logvar).exp() * torch.randn_like(mu) if sample_latent else mu
        pred = self.decode_logits(z).argmax(-1)
        self.train(was_training)
        ok = (pred == tokens)
        return ok.float().mean().item(), ok.all(dim=-1).float().mean().item()


# ----------------------------------------------------------------------------------------------------
# Stage 2: the backbone (one position per chunk) and the energy head
# ----------------------------------------------------------------------------------------------------
class LocalMix(nn.Module):
    """OPTIONAL (USE_LOCAL_MIX): one small self-attention layer over the K tokens INSIDE one chunk, run before
    the flatten-and-project input step, so the tokens of a chunk can interact before they are squeezed into one
    vector. It is not in the CALM paper; it is our own addition and is off by default."""

    def __init__(self, d, heads=8, k=CHUNK_K):
        super().__init__()
        self.heads = heads
        self.norm = make_norm(d, NORM_TYPE)
        self.pos = nn.Parameter(torch.randn(k, d) * 0.02)
        self.qkv = nn.Linear(d, 3 * d)
        self.out = nn.Linear(d, d)

    def forward(self, x):
        """x: (..., K, d) -> (..., K, d)."""
        lead, k, d = x.shape[:-2], x.shape[-2], x.shape[-1]
        y = (self.norm(x) + self.pos).reshape(-1, k, d)
        q, kk, v = self.qkv(y).view(-1, k, 3, self.heads, d // self.heads).permute(2, 0, 3, 1, 4)
        o = F.scaled_dot_product_attention(q, kk, v)                      # no causal mask: all K tokens are inputs
        o = o.transpose(1, 2).reshape(-1, k, d)
        return x + self.out(o).reshape(*lead, k, d)


class ChunkInput(nn.Module):
    """The "input embed + MLP" block: K token ids (one chunk) -> ONE vector of D_MODEL numbers.
    K lookups in a V x EMB_D table -> (optional local attention) -> flatten to K*EMB_D -> Linear -> GELU -> Linear."""

    def __init__(self, vocab_size, k, emb_d, d_model):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_d)
        self.local = LocalMix(emb_d, k=k) if USE_LOCAL_MIX else None
        self.fc1 = nn.Linear(k * emb_d, d_model)
        self.fc2 = nn.Linear(d_model, d_model)

    def forward(self, tokens):
        """tokens: (..., K) long -> (..., d_model)."""
        e = self.emb(tokens)
        if self.local is not None:
            e = self.local(e)
        return self.fc2(F.gelu(self.fc1(e.flatten(-2))))


class CALMBlock(nn.Module):
    """Pre-norm decoder block: x + GQA(norm(x)), then x + FFN(norm(x)). decode_step() mirrors forward() for one position."""

    def __init__(self, d_model, n_heads, n_kv_heads, d_ff):
        super().__init__()
        self.attn = CachedGQAttention(d_model, n_heads, n_kv_heads)
        self.ff = FFN(d_model, d_ff)
        self.norm1 = make_norm(d_model, NORM_TYPE)
        self.norm2 = make_norm(d_model, NORM_TYPE)

    def forward(self, x, key_padding_mask=None, k_cache=None, v_cache=None, rope=None):
        x = x + self.attn(self.norm1(x), key_padding_mask=key_padding_mask, k_cache=k_cache, v_cache=v_cache, rope=rope)
        return x + self.ff(self.norm2(x))

    def decode_step(self, x_t, k_cache, v_cache, cache_len, key_mask=None, rope=None):
        x_t = x_t + self.attn.decode_step(self.norm1(x_t), k_cache, v_cache, cache_len, key_mask, rope)
        return x_t + self.ff(self.norm2(x_t))


class CALMBackbone(nn.Module):
    """The transformer that works at CHUNK level. Position i holds ONE vector standing for the K tokens of
    chunk i; its output h_i (D_MODEL numbers) is used to guess the latent of chunk i+1. RoPE positions are
    chunk indices. There is no vocabulary-sized output layer anywhere in this model."""

    def __init__(self, vocab_size, k, emb_d, d_model, n_heads, n_layers, d_ff, max_chunks):
        super().__init__()
        self.k = k
        self.chunk_in = ChunkInput(vocab_size, k, emb_d, d_model)
        cos, sin = build_rope_cache(max_chunks, d_model // n_heads, ROPE_BASE)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        n_kv_heads = max(1, n_heads // GQA_GROUP_SIZE)
        self.blocks = nn.ModuleList([CALMBlock(d_model, n_heads, n_kv_heads, d_ff) for _ in range(n_layers)])
        self.final_norm = make_norm(d_model, NORM_TYPE)

    def _rope(self, position_ids):
        cos, sin = self.rope_cos[position_ids], self.rope_sin[position_ids]    # (batch or 1, seq, head_dim)
        return cos.unsqueeze(1), sin.unsqueeze(1)

    def forward(self, chunk_tokens, key_padding_mask=None, position_ids=None):
        """Training / full-recompute forward. chunk_tokens: (batch, P, K) long -> h: (batch, P, d_model)."""
        batch_size, num_chunks, _ = chunk_tokens.shape
        if position_ids is None:
            position_ids = torch.arange(num_chunks, device=chunk_tokens.device).unsqueeze(0)
        h = self.chunk_in(chunk_tokens)
        rope = self._rope(position_ids)
        for block in self.blocks:
            h = block(h, key_padding_mask=key_padding_mask, rope=rope)
        return self.final_norm(h)

    def prefill(self, chunk_tokens, pad_lens, cache):
        """PREFILL. chunk_tokens: (batch, P, K), LEFT-padded with whole pad chunks; pad_lens: list of ints (pad
        CHUNKS per row). Fills cache positions [0:P] of every layer and returns h at the last position (batch, d_model)."""
        batch_size, prefill_len, _ = chunk_tokens.shape
        assert prefill_len <= cache.max_len, "prompt window longer than the cache"
        pad_t = torch.tensor(pad_lens, dtype=torch.long, device=chunk_tokens.device)
        arange = torch.arange(prefill_len, device=chunk_tokens.device).unsqueeze(0)
        position_ids = (arange - pad_t.unsqueeze(1)).clamp(min=0)            # first REAL chunk -> position 0
        key_padding_mask = (arange >= pad_t.unsqueeze(1)) if any(p > 0 for p in pad_lens) else None
        h = self.chunk_in(chunk_tokens)
        rope = self._rope(position_ids)
        for layer_idx, block in enumerate(self.blocks):
            h = block(h, key_padding_mask=key_padding_mask,
                      k_cache=cache.k[layer_idx], v_cache=cache.v[layer_idx], rope=rope)
        cache.length = prefill_len
        return self.final_norm(h[:, -1, :])

    def decode_step(self, chunk_tokens, pad_t, has_padding, cache):
        """DECODE. chunk_tokens: (batch, K) -- ONE new chunk per row. Appends to the cache and returns h (batch, d_model)
        for guessing the chunk after it."""
        cache_len = cache.length
        assert cache_len < cache.max_len, "cache is full -- re-prefill before decoding further"
        position_ids = (cache_len - pad_t).clamp(min=0)                      # (batch,)
        key_mask = None
        if has_padding:
            key_mask = torch.arange(cache_len + 1, device=chunk_tokens.device).unsqueeze(0) >= pad_t.unsqueeze(1)
        h = self.chunk_in(chunk_tokens)
        rope = (self.rope_cos[position_ids], self.rope_sin[position_ids])    # each (batch, head_dim)
        for layer_idx, block in enumerate(self.blocks):
            h = block.decode_step(h, cache.k[layer_idx], cache.v[layer_idx], cache_len, key_mask, rope)
        cache.length = cache_len + 1
        return self.final_norm(h)


class EnergyHead(nn.Module):
    """The generative head. Input: the backbone's output h (D_MODEL numbers) plus random noise; output: one
    GUESS of the next chunk's latent (LATENT_DIM numbers). Calling it N times with fresh noise gives N different
    guesses -- that randomness is what lets the model express "several continuations are plausible".
        x = Linear(h) + Linear(noise)   ->   HEAD_LAYERS x (x + FFN(norm(x)))   ->   norm -> Linear -> latent"""

    def __init__(self, d_model, head_d, layers, noise_dim, latent_dim):
        super().__init__()
        self.noise_dim = noise_dim
        self.h_proj = nn.Linear(d_model, head_d)
        self.n_proj = nn.Linear(noise_dim, head_d, bias=False)
        self.blocks = nn.ModuleList([ResFFN(head_d, 2 * head_d) for _ in range(layers)])
        self.norm = make_norm(head_d, NORM_TYPE)
        self.out = nn.Linear(head_d, latent_dim)

    def forward(self, h, n_samples):
        """h: (..., d_model) -> guesses (..., n_samples, latent_dim)."""
        eps = torch.rand(*h.shape[:-1], n_samples, self.noise_dim, device=h.device, dtype=h.dtype) - 0.5   # uniform [-0.5, 0.5]
        x = self.h_proj(h).unsqueeze(-2) + self.n_proj(eps)
        for block in self.blocks:
            x = block(x)
        return self.out(self.norm(x))


def _pair_dist(a, b):
    """Euclidean distances between every vector of a (P, N, l) and every vector of b (P, M, l) -> (P, N, M), using
    one batched matrix product (a memory-friendly form of cdist whose gradient is safe at zero distance)."""
    d2 = (a.pow(2).sum(-1).unsqueeze(2) + b.pow(2).sum(-1).unsqueeze(1) - 2.0 * torch.bmm(a, b.transpose(1, 2)))
    return d2.clamp(min=1e-8).sqrt()


def energy_score_loss(guesses, mu, logvar, n_targets):
    """The CALM training loss -- a likelihood-free, strictly proper scoring rule (the energy score):

        loss = 2 * mean_{n,m} ||z_m - zhat_n||  -  1/(N(N-1)) * sum_{n != k} ||zhat_n - zhat_k||

    guesses: (P, N, l) -- N random guesses of the next latent for each of P positions.
    mu, logvar: (P, l) -- the frozen autoencoder's posterior for the REAL next chunk; M = n_targets samples z_m are
    drawn from it. First term: pull every guess towards the real latent. Second term: push the guesses away from
    each other, so the head keeps a spread of different guesses instead of collapsing to one.
    Returns (loss averaged over positions, attract term, repel term)."""
    P, N, l = guesses.shape
    targets = mu.unsqueeze(1) + (0.5 * logvar).exp().unsqueeze(1) * torch.randn(P, n_targets, l, device=mu.device, dtype=mu.dtype)
    attract = _pair_dist(guesses, targets).mean(dim=(1, 2))                       # (P,)
    off_diagonal = 1.0 - torch.eye(N, device=guesses.device, dtype=guesses.dtype)
    repel = (_pair_dist(guesses, guesses) * off_diagonal).sum(dim=(1, 2)) / (N * (N - 1))
    loss = (2.0 * attract - repel).mean()
    return loss, attract.mean().detach(), repel.mean().detach()


class CALMLM(nn.Module):
    """Backbone + energy head. forward() is the TRAINING step on a batch of token windows."""

    def __init__(self, vocab_size, autoencoder):
        super().__init__()
        self.backbone = CALMBackbone(vocab_size, CHUNK_K, EMB_D, D_MODEL, N_HEADS, N_LAYERS, D_FF, CONTEXT_LEN // CHUNK_K)
        self.head = EnergyHead(D_MODEL, HEAD_D, HEAD_LAYERS, NOISE_DIM, LATENT_DIM)
        self._ae = [autoencoder]   # kept OUT of the module tree on purpose: frozen, not trained, not in the checkpoint

    @property
    def ae(self):
        return self._ae[0]

    def forward(self, windows):
        """windows: (batch, T) token ids with T a multiple of K. Position i sees chunks 0..i and guesses chunk i+1."""
        batch_size, seq_len = windows.shape
        chunks = windows.view(batch_size, seq_len // CHUNK_K, CHUNK_K)
        with torch.no_grad():
            mu, logvar = self.ae.encode(chunks[:, 1:])                   # real latents of chunks 1..C-1 (frozen autoencoder)
        h = self.backbone(chunks[:, :-1])                                # (batch, C-1, d_model)
        guesses = self.head(h, ENERGY_SAMPLES)                           # (batch, C-1, N, l)
        P = batch_size * (chunks.shape[1] - 1)
        return energy_score_loss(guesses.reshape(P, ENERGY_SAMPLES, LATENT_DIM), mu.reshape(P, LATENT_DIM),
                                 logvar.reshape(P, LATENT_DIM), ENERGY_TARGETS)

## 9. Build the models

Builds the autoencoder and the language model, prints the parameter breakdown, and times a few steps of each stage (speed probe).

In [14]:
vocab_size = bpe_tokenizer.get_vocab_size()
TOKENS_PER_WORD = len(all_word_ids) / word_count
print(f"Corpus: {word_count:,} words = {len(all_word_ids):,} tokens ({TOKENS_PER_WORD:.2f} tokens per word) "
      f"-> {len(all_word_ids) // CHUNK_K:,} chunks of {CHUNK_K} tokens")

torch.manual_seed(0)
ae = ChunkAutoencoder(vocab_size, CHUNK_K, AE_D, AE_FF, LATENT_DIM).to(device)
model = CALMLM(vocab_size, ae).to(device)      # backbone + energy head; `ae` is held outside it (frozen in Stage 2)


def _n(*mods):
    return sum(p.numel() for m in mods for p in m.parameters())


bb = model.backbone
n_ae, n_model = _n(ae), _n(model)
n_dec_only = n_ae - _n(ae.enc_tok, ae.enc_merge, ae.enc_mix, ae.enc_norm, ae.enc_out)   # what generation needs from the autoencoder
print(f"\nStage 2 model (backbone + head): {n_model:,} parameters ({n_model / 1e6:.1f}M)")
print(f"  input embedding table {_n(bb.chunk_in.emb) / 1e6:.1f}M | input MLP{' + local mix' if USE_LOCAL_MIX else ''} "
      f"{(_n(bb.chunk_in) - _n(bb.chunk_in.emb)) / 1e6:.1f}M | {N_LAYERS} layers: attention "
      f"{_n(*[b.attn for b in bb.blocks]) / 1e6:.1f}M + FFN {_n(*[b.ff for b in bb.blocks]) / 1e6:.1f}M | "
      f"energy head {_n(model.head) / 1e6:.1f}M")
print(f"Stage 1 autoencoder: {n_ae:,} parameters ({n_ae / 1e6:.1f}M); the part generation needs (decoder + tied table): {n_dec_only / 1e6:.1f}M")
total_params = n_ae + n_model
print(f"TOTAL (counting everything, encoder included): {total_params:,} ({total_params / 1e6:.1f}M)")
if not (50_000_000 <= total_params <= 300_000_000):
    print("WARNING: this is outside the required 50M-300M range! Change D_FF / N_LAYERS / AE_D above.")
print(f"Positions per training window: {CONTEXT_LEN // CHUNK_K - 1} (the baseline has {CONTEXT_LEN - 1}); "
      f"no vocabulary-sized output layer in the backbone")

# ---- checkpoint names: they record every setting that changes a weight's shape or meaning ----------------------
DATA_TAG = f"{round(word_count / 1e6)}Mw" if word_count >= 1_000_000 else f"{word_count // 1000}kw"
_cfg = f"K{CHUNK_K}_l{LATENT_DIM}"
# "a1bpe" = the from-scratch Assignment-1 BPE; a checkpoint trained with any other tokenizer has different token ids.
AE_CHECKPOINT_PATH = f"/kaggle/working/calm_ae_{mode_tag}_{_cfg}_d{AE_D}_{DATA_TAG}_a1bpe_{RUN_TAG}_checkpoint.pt"
CHECKPOINT_PATH = (f"/kaggle/working/calm_lm_{mode_tag}_{_cfg}_L{N_LAYERS}_d{D_MODEL}_ff{D_FF}_{DATA_TAG}_a1bpe_{RUN_TAG}_checkpoint.pt")
print(f"Autoencoder checkpoint: {os.path.basename(AE_CHECKPOINT_PATH)}")
print(f"Language-model checkpoint: {os.path.basename(CHECKPOINT_PATH)}")

# ---- KV cache memory: chunk positions vs the token-level baseline ----------------------------------------------------
_attn0 = bb.blocks[0].attn
_bytes_per_pos = 2 * len(bb.blocks) * _attn0.n_kv_heads * _attn0.head_dim * 2     # K and V, fp16
print(f"\nKV cache per sequence at {CONTEXT_LEN} tokens ({CONTEXT_LEN // CHUNK_K} chunk positions, fp16): "
      f"{_bytes_per_pos * (CONTEXT_LEN // CHUNK_K) / 1e6:.1f} MB")
log_event("model_built", ae_params=n_ae, model_params=n_model, total_params=total_params, chunk_k=CHUNK_K, latent_dim=LATENT_DIM,
          ae_d=AE_D, d_model=D_MODEL, n_layers=N_LAYERS, n_heads=N_HEADS, gqa_group=GQA_GROUP_SIZE, d_ff=D_FF,
          energy_samples=ENERGY_SAMPLES, energy_targets=ENERGY_TARGETS, local_mix=USE_LOCAL_MIX, tokens_per_word=round(TOKENS_PER_WORD, 3))

Corpus: 240,000,000 words = 276,362,790 tokens (1.15 tokens per word) -> 69,090,697 chunks of 4 tokens

Stage 2 model (backbone + head): 171,978,880 parameters (172.0M)
  input embedding table 30.7M | input MLP 3.0M | 12 layers: attention 21.3M + FFN 113.3M | energy head 3.7M
Stage 1 autoencoder: 31,182,080 parameters (31.2M); the part generation needs (decoder + tied table): 25.8M
TOTAL (counting everything, encoder included): 203,160,960 (203.2M)
Positions per training window: 511 (the baseline has 2047); no vocabulary-sized output layer in the backbone
Autoencoder checkpoint: calm_ae_full_K4_l128_d512_240Mw_a1bpe_calm240Mw_1ep_checkpoint.pt
Language-model checkpoint: calm_lm_full_K4_l128_L12_d768_ff6144_240Mw_a1bpe_calm240Mw_1ep_checkpoint.pt

KV cache per sequence at 2048 tokens (512 chunk positions, fp16): 9.4 MB


/tmp/ipykernel_23/2161731718.py:103: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


In [15]:
# ---- Speed probe: time a few real training steps of each stage -> projected hours for the full run -----------------
import time


def _time_steps(step_fn, n=6, warm=2):
    """Average seconds per call of step_fn() after `warm` warm-up calls. Peak GPU memory is reset first."""
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()
    for i in range(warm + n):
        if i == warm:
            if device == "cuda":
                torch.cuda.synchronize()
            t0 = time.time()
        step_fn()
    if device == "cuda":
        torch.cuda.synchronize()
    peak = torch.cuda.max_memory_allocated() / 1e9 if device == "cuda" else float("nan")
    return (time.time() - t0) / n, peak


_probe_rows = train_data[torch.randint(0, len(train_data), (BATCH_SIZE,))].long().to(device)
_probe_chunks = train_data[:2].reshape(-1, CHUNK_K).long().to(device)[:AE_BATCH_CHUNKS]
# throwaway optimisers with lr = 0: the real weights are not changed by the probe
_ae_probe_opt = torch.optim.AdamW(ae.parameters(), lr=0.0, fused=USE_FUSED_ADAMW)
_lm_probe_opt = torch.optim.AdamW(model.parameters(), lr=0.0, fused=USE_FUSED_ADAMW)


def _ae_step():
    _ae_probe_opt.zero_grad()
    loss, _ = ae(_probe_chunks)
    loss.backward()
    _ae_probe_opt.step()


def _lm_step():
    _lm_probe_opt.zero_grad()
    loss, _, _ = model(_probe_rows)
    loss.backward()
    _lm_probe_opt.step()


ae.train(); model.train()
if len(_probe_chunks) < AE_BATCH_CHUNKS:
    _probe_chunks = _probe_chunks.repeat(-(-AE_BATCH_CHUNKS // len(_probe_chunks)), 1)[:AE_BATCH_CHUNKS]
ae_sec, ae_mem = _time_steps(_ae_step)
lm_sec, lm_mem = _time_steps(_lm_step)
_lm_win = len(train_data)
_ae_win = len(train_data) if AE_WORDS >= word_count else min(len(train_data), max(1, int(AE_WORDS * TOKENS_PER_WORD / CONTEXT_LEN)))
_ae_max = max(1, -(-(_ae_win * (CONTEXT_LEN // CHUNK_K)) // AE_BATCH_CHUNKS))
_lm_steps = -(-_lm_win // BATCH_SIZE) * NUM_EPOCHS
print(f"Stage 1 (autoencoder): {ae_sec:.2f} s/step ({AE_BATCH_CHUNKS} chunks), peak GPU memory {ae_mem:.1f} GB. "
      f"One pass over {_ae_win:,} windows = {_ae_max:,} steps = at most {ae_sec * _ae_max / 3600:.1f} h (it stops earlier when it reaches {AE_TARGET_ACC:.1%}).")
print(f"Stage 2 (language model): {lm_sec:.2f} s/step ({BATCH_SIZE} windows = {BATCH_SIZE * (CONTEXT_LEN - CHUNK_K):,} predicted tokens), "
      f"peak GPU memory {lm_mem:.1f} GB. {_lm_steps:,} steps = {lm_sec * _lm_steps / 3600:.1f} h.")
print("If Stage 2 would not fit in your 12 h Kaggle session: the notebook checkpoints and resumes, so you can continue it in a second "
      "session (or account); or lower MAX_WORDS_FOR_TRAINING.")
log_event("speed_probe", ae_sec_per_step=round(ae_sec, 3), ae_peak_gb=round(ae_mem, 2), ae_max_steps=_ae_max,
          lm_sec_per_step=round(lm_sec, 3), lm_peak_gb=round(lm_mem, 2), lm_steps=_lm_steps,
          projected_hours_ae_max=round(ae_sec * _ae_max / 3600, 2), projected_hours_lm=round(lm_sec * _lm_steps / 3600, 2))
del _ae_probe_opt, _lm_probe_opt

Stage 1 (autoencoder): 0.19 s/step (1024 chunks), peak GPU memory 3.3 GB. One pass over 67,471 windows = 33,736 steps = at most 1.8 h (it stops earlier when it reaches 99.9%).
Stage 2 (language model): 1.15 s/step (8 windows = 16,352 predicted tokens), peak GPU memory 8.1 GB. 16,700 steps = 5.4 h.
If Stage 2 would not fit in your 12 h Kaggle session: the notebook checkpoints and resumes, so you can continue it in a second session (or account); or lower MAX_WORDS_FOR_TRAINING.


/tmp/ipykernel_23/2161731718.py:103: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


## 10. Stage 1: train the autoencoder

Trains on up to `AE_WORDS` words and **stops early** once held-out per-token reconstruction accuracy (decoding a *sampled* latent) reaches `AE_TARGET_ACC`. Checkpoints and resumes like the language model; if a finished autoencoder checkpoint is found (locally or in the bucket) this step is skipped. Afterwards the autoencoder is frozen.

In [16]:
import math

# ---- what Stage 1 trains on ------------------------------------------------------------------------------------------
n_ae_windows = len(train_data) if AE_WORDS >= word_count else min(len(train_data), max(1, int(AE_WORDS * TOKENS_PER_WORD / CONTEXT_LEN)))
ae_train_windows = train_data[:n_ae_windows]           # int32 (windows, CONTEXT_LEN); the data is already shuffled across sources
CHUNKS_PER_WINDOW = CONTEXT_LEN // CHUNK_K
AE_MAX_STEPS = max(1, -(-(n_ae_windows * CHUNKS_PER_WINDOW) // AE_BATCH_CHUNKS))   # one pass over its windows
ae_min_steps = min(AE_MIN_STEPS, AE_MAX_STEPS) if not QUICK_TEST else min(30, AE_MAX_STEPS)
ae_eval_every = AE_EVAL_EVERY if not QUICK_TEST else 40
AE_CKPT_EVERY_EVALS = 1 if QUICK_TEST else 2           # push the checkpoint to the Hub every this many evaluations
ae_eval_chunks = val_data[:max(1, min(len(val_data), 40))].reshape(-1, CHUNK_K).long().to(device)   # held-out chunks (never trained on)
print(f"Stage 1 data: {n_ae_windows:,} windows = {n_ae_windows * CHUNKS_PER_WINDOW:,} chunks "
      f"(~{n_ae_windows * CONTEXT_LEN / TOKENS_PER_WORD / 1e6:.0f}M words). Up to {AE_MAX_STEPS:,} steps of {AE_BATCH_CHUNKS} chunks; "
      f"stops once held-out token accuracy >= {AE_TARGET_ACC:.1%} (after at least {ae_min_steps} steps). Held-out chunks: {len(ae_eval_chunks):,}")


def ae_heldout_accuracy():
    """Per-token and whole-chunk reconstruction accuracy on the held-out chunks, decoding a SAMPLED latent and the posterior mean."""
    out = {"sampled": [0.0, 0.0], "mean": [0.0, 0.0]}
    slices = list(range(0, len(ae_eval_chunks), 2048))
    for key, sample in (("sampled", True), ("mean", False)):
        for s in slices:
            part = ae_eval_chunks[s:s + 2048]
            tok, chunk = ae.reconstruction_accuracy(part, sample_latent=sample)
            out[key][0] += tok * len(part)
            out[key][1] += chunk * len(part)
        out[key] = [v / len(ae_eval_chunks) for v in out[key]]
    return out


def ae_lr(step):
    warm = min(1.0, (step + 1) / AE_WARMUP_STEPS)
    cosine = 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(1.0, step / AE_MAX_STEPS)))
    return AE_LR * warm * cosine


ae_optimizer = torch.optim.AdamW(ae.parameters(), lr=AE_LR, fused=USE_FUSED_ADAMW)
ae_step, ae_done = 0, False
try_pull_checkpoint_from_hub(AE_CHECKPOINT_PATH)
if os.path.exists(AE_CHECKPOINT_PATH):
    _ck = torch.load(AE_CHECKPOINT_PATH, map_location=device)
    ae.load_state_dict(_ck["model"])
    ae_optimizer.load_state_dict(_ck["optimizer"])
    ae_step, ae_done = _ck["step"], _ck.get("done", False)
    print(f"Found an autoencoder checkpoint: step {ae_step}, {'FINISHED' if ae_done else 'resuming'}")
    del _ck
else:
    print("No autoencoder checkpoint found. Starting Stage 1 from scratch.")


def save_ae(done):
    tmp = AE_CHECKPOINT_PATH + ".tmp"
    torch.save({"model": ae.state_dict(), "optimizer": ae_optimizer.state_dict(), "step": ae_step, "done": done}, tmp)
    os.replace(tmp, AE_CHECKPOINT_PATH)


if not ae_done:
    ae.train()
    t0, step0, n_evals, running = time.time(), ae_step, 0, []
    log_event("ae_start", from_step=ae_step, max_steps=AE_MAX_STEPS, windows=n_ae_windows)
    while ae_step < AE_MAX_STEPS:
        g = torch.Generator().manual_seed(100_000 + ae_step)
        pool = ae_train_windows[torch.randint(0, n_ae_windows, (16,), generator=g)].reshape(-1, CHUNK_K)   # chunks of 16 windows...
        batch = pool[torch.randint(0, len(pool), (AE_BATCH_CHUNKS,), generator=g)].long().to(device)        # ...sampled at random
        for group in ae_optimizer.param_groups:
            group["lr"] = ae_lr(ae_step)
        loss, stats = ae(batch)
        ae_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(ae.parameters(), 1.0)
        ae_optimizer.step()
        ae_step += 1
        running.append((loss.item(), stats["rec"].item(), stats["tok_acc"].item(), stats["kl"].item()))
        if ae_step % 100 == 0:
            a = [sum(x[i] for x in running) / len(running) for i in range(4)]
            running = []
            rate = (ae_step - step0) / max(1e-9, time.time() - t0)
            print(f"  AE step {ae_step}/{AE_MAX_STEPS}: loss {a[0]:.3f} (rec {a[1]:.3f}, KL/dim {a[3]:.3f})  train token acc {a[2]:.4f}  "
                  f"({rate:.2f} step/s, <= {(AE_MAX_STEPS - ae_step) / max(1e-9, rate) / 3600:.1f} h left)")
        if ae_step % ae_eval_every == 0 or ae_step == AE_MAX_STEPS:
            acc = ae_heldout_accuracy()
            n_evals += 1
            print(f"  >> held-out @ step {ae_step}: token acc {acc['sampled'][0]:.4f} / chunk acc {acc['sampled'][1]:.4f} (sampled latent); "
                  f"{acc['mean'][0]:.4f} / {acc['mean'][1]:.4f} (posterior mean)")
            log_event("ae_eval", step=ae_step, tok_acc_sampled=round(acc["sampled"][0], 5), chunk_acc_sampled=round(acc["sampled"][1], 5),
                      tok_acc_mean=round(acc["mean"][0], 5), chunk_acc_mean=round(acc["mean"][1], 5))
            reached = acc["sampled"][0] >= AE_TARGET_ACC and ae_step >= ae_min_steps
            save_ae(done=reached)
            if reached or n_evals % AE_CKPT_EVERY_EVALS == 0:
                try:
                    push_checkpoint_to_hub(AE_CHECKPOINT_PATH)
                except Exception as e:
                    print(f"(autoencoder checkpoint push failed: {type(e).__name__} -- local copy is fine)")
                sync_run_log()
            if reached:
                ae_done = True
                print(f"Target reached ({acc['sampled'][0]:.4f} >= {AE_TARGET_ACC}) -- Stage 1 finished early at step {ae_step}.")
                break
            ae.train()
    if not ae_done:
        save_ae(done=True)
        ae_done = True
        print(f"WARNING: Stage 1 used its whole budget ({AE_MAX_STEPS} steps) without reaching {AE_TARGET_ACC:.1%} held-out token accuracy "
              f"(last: see above). Stage 2 will still run, but the autoencoder caps the quality of every generated token. "
              f"Options: raise AE_WORDS, lower AE_TARGET_ACC, or train longer (delete the checkpoint's 'done' flag by re-running with a bigger AE_WORDS).")
    try:
        push_checkpoint_to_hub(AE_CHECKPOINT_PATH)
    except Exception as e:
        print(f"(autoencoder checkpoint push failed: {type(e).__name__})")
    log_event("ae_end", steps=ae_step, minutes=round((time.time() - t0) / 60, 1))
    sync_run_log()

# ---- freeze it: Stage 2 only uses the autoencoder, it never changes it --------------------------------------------------
ae.eval()
for p in ae.parameters():
    p.requires_grad_(False)
acc = ae_heldout_accuracy()
print(f"\nAutoencoder frozen. Held-out: token accuracy {acc['sampled'][0]:.4f} (sampled latent) / {acc['mean'][0]:.4f} (posterior mean); "
      f"whole-chunk accuracy {acc['sampled'][1]:.4f} / {acc['mean'][1]:.4f}")
print("Example (a real held-out chunk of 4 tokens and its reconstruction):")
with torch.no_grad():
    _ex = ae_eval_chunks[:3]
    _mu, _lv = ae.encode(_ex)
    _rec = ae.decode_logits(_mu).argmax(-1)
for a_, b_ in zip(_ex.tolist(), _rec.tolist()):
    print("  real  :", decode(a_), "|", a_)
    print("  recon :", decode(b_), "|", b_)
log_event("ae_frozen", tok_acc_sampled=round(acc["sampled"][0], 5), chunk_acc_sampled=round(acc["sampled"][1], 5),
          tok_acc_mean=round(acc["mean"][0], 5), chunk_acc_mean=round(acc["mean"][1], 5))

Stage 1 data: 67,471 windows = 34,545,152 chunks (~120M words). Up to 33,736 steps of 1024 chunks; stops once held-out token accuracy >= 99.9% (after at least 3000 steps). Held-out chunks: 20,480
No matching checkpoint on the Hub yet (FileNotFoundError) -- starting fresh.
No autoencoder checkpoint found. Starting Stage 1 from scratch.


/tmp/ipykernel_23/2161731718.py:103: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


  AE step 100/33736: loss 35.399 (rec 35.326, KL/dim 0.456)  train token acc 0.1370  (5.14 step/s, <= 1.8 h left)
  AE step 200/33736: loss 19.702 (rec 19.497, KL/dim 1.602)  train token acc 0.3962  (5.09 step/s, <= 1.8 h left)
  AE step 300/33736: loss 13.422 (rec 13.130, KL/dim 2.277)  train token acc 0.6270  (5.03 step/s, <= 1.8 h left)
  AE step 400/33736: loss 9.706 (rec 9.417, KL/dim 2.255)  train token acc 0.7469  (4.97 step/s, <= 1.9 h left)
  AE step 500/33736: loss 7.675 (rec 7.411, KL/dim 2.062)  train token acc 0.7978  (4.90 step/s, <= 1.9 h left)
  AE step 600/33736: loss 6.592 (rec 6.343, KL/dim 1.944)  train token acc 0.8240  (4.84 step/s, <= 1.9 h left)
  AE step 700/33736: loss 5.824 (rec 5.589, KL/dim 1.836)  train token acc 0.8416  (4.81 step/s, <= 1.9 h left)
  AE step 800/33736: loss 5.376 (rec 5.153, KL/dim 1.743)  train token acc 0.8504  (4.79 step/s, <= 1.9 h left)
  AE step 900/33736: loss 5.074 (rec 4.858, KL/dim 1.682)  train token acc 0.8562  (4.78 step/s, <

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

... (17 outputs left out)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            


Autoencoder frozen. Held-out: token accuracy 0.9991 (sampled latent) / 0.9991 (posterior mean); whole-chunk accuracy 0.9964 / 0.9965
Example (a real held-out chunk of 4 tokens and its reconstruction):
  real  : ] ○ | [523, 204, 144, 132]
  recon : ] ○ | [523, 204, 144, 132]
  real  :  Maharashtra State Electricity | [53, 3932, 464, 4687]
  recon :  Maharashtra State Electricity | [53, 3932, 464, 4687]
  real  : Distribution Co. Ltd. [ | [8005, 3943, 1785, 84]
  recon : Distribution Co. Ltd. [ | [8005, 3943, 1785, 84]


### Does the autoencoder generalise to unseen text?

Reconstruction accuracy on the **whole held-out documents** (never trained on), overall and split by how common the tokens are. A big gap between common and rare tokens means rare words are the weak spot.

In [17]:
# ---- Does the frozen autoencoder generalise to text it has never seen? ---------------------------------------------------
# The accuracy above uses chunks cut from the SAME documents as the training windows (a random 1% slice). The honest test is
# the WHOLE held-out documents (never trained on, never used for the tokenizer; the same ones as in the other notebooks),
# split by how common the tokens are -- rare tokens have under-trained embeddings and are where reconstruction should suffer.
def ae_unseen_text_report(max_chunks=200_000):
    if not HELDOUT_FILES:
        print("No held-out documents on disk (QUICK_TEST mode) -- skipping.")
        return None
    # how often each token occurs in the training corpus (sampled from the first ~20M tokens to stay fast)
    sample = torch.frombuffer(all_word_ids, dtype=torch.int32)[:20_000_000].long() if isinstance(all_word_ids, array.array) else torch.tensor(all_word_ids[:20_000_000])
    freq = torch.bincount(sample, minlength=bpe_tokenizer.get_vocab_size()).float() * (len(all_word_ids) / len(sample))
    bins = [("very rare (< 100 uses)", 0, 100), ("rare (100 - 1,000)", 100, 1_000), ("common (1,000 - 100,000)", 1_000, 100_000), ("very common (>= 100,000)", 100_000, float("inf"))]
    rows, per_source = {name: [0, 0] for name, _, _ in bins}, {}
    rng_ = random.Random(5)
    for source, path in HELDOUT_FILES.items():
        ids = []
        with open(path, "r", encoding="utf-8") as f:
            docs = [line for line in f if line.strip()]
        rng_.shuffle(docs)
        for doc in docs[:300]:
            ids.extend(encode(doc))
            if len(ids) >= max_chunks * CHUNK_K // len(HELDOUT_FILES):
                break
        ids = ids[:len(ids) // CHUNK_K * CHUNK_K]
        chunks = torch.tensor(ids, dtype=torch.long).view(-1, CHUNK_K)
        ok_tok, n_tok = 0, 0
        with torch.no_grad():
            for s in range(0, len(chunks), 2048):
                part = chunks[s:s + 2048].to(device)
                mu, lv = ae.encode(part)
                z = mu + (0.5 * lv).exp() * torch.randn_like(mu)          # a sampled latent, like a (good) language-model guess
                ok = (ae.decode_logits(z).argmax(-1) == part)
                f_ = freq[part.cpu()]
                for name, lo, hi in bins:
                    sel = ((f_ >= lo) & (f_ < hi)).to(ok.device)
                    rows[name][0] += ok[sel].sum().item(); rows[name][1] += sel.sum().item()
                ok_tok += ok.sum().item(); n_tok += ok.numel()
        per_source[source] = ok_tok / max(1, n_tok)
        print(f"  held-out {source:14s}: token accuracy {per_source[source]:.4f}  ({n_tok:,} tokens)")
    print("  by token frequency (all held-out sources together):")
    for name, _, _ in bins:
        c, n = rows[name]
        print(f"    {name:28s}: {c / max(1, n):.4f}   ({n:,} tokens)")
    log_event("ae_unseen_text", per_source={k: round(v, 5) for k, v in per_source.items()},
              by_frequency={k: [round(v[0] / max(1, v[1]), 5), v[1]] for k, v in rows.items()})
    return per_source, rows


import array, random
_ae_unseen = ae_unseen_text_report()

  held-out acts          : token accuracy 0.9997  (201,544 tokens)
  held-out high_courts   : token accuracy 0.9998  (201,012 tokens)
  held-out supreme_court : token accuracy 0.9999  (202,864 tokens)
  held-out tribunals     : token accuracy 0.9996  (200,396 tokens)
  by token frequency (all held-out sources together):
    very rare (< 100 uses)      : 0.9974   (45,104 tokens)
    rare (100 - 1,000)          : 0.9989   (53,231 tokens)
    common (1,000 - 100,000)    : 0.9999   (343,940 tokens)
    very common (>= 100,000)    : 1.0000   (363,541 tokens)


/tmp/ipykernel_23/2161731718.py:103: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


## 11. Stage 2: train the language model

Same resumable epoch/batch loop as the other notebooks, plus (our additions, all standard): a linear warmup and cosine decay of the
learning rate (a function of the global step, so resuming is exact) and gradient-norm clipping. Plain fp32 throughout. The printed loss is
the **energy loss** (attract term minus repel term); it is *not* comparable to the cross-entropy of the other notebooks.

In [18]:
optimizer = build_optimizer(model, OPTIMIZER_TYPE, LEARNING_RATE)
start_epoch = 0          # first epoch still to (finish) train
resume_batch = 0         # batches of start_epoch that are already done
resume_loss_sum = 0.0    # their summed loss (only for the running average that is printed)

try_pull_checkpoint_from_hub(CHECKPOINT_PATH)  # in case a fresh session/account has no local copy

if os.path.exists(CHECKPOINT_PATH):
    print("Found an existing Stage-2 checkpoint, resuming from it...")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    start_epoch = checkpoint["epoch"] + 1           # "epoch" = last epoch that was COMPLETED (-1 = none yet)
    if checkpoint.get("in_progress_epoch") == start_epoch:
        resume_batch = checkpoint["next_batch"]
        resume_loss_sum = checkpoint.get("loss_sum", 0.0)
        print(f"Resuming in the middle of epoch {start_epoch}, at batch {resume_batch}")
    else:
        print(f"Resuming from the start of epoch {start_epoch}")
    log_event("resume", start_epoch=start_epoch, resume_batch=resume_batch)
    del checkpoint
else:
    print("No Stage-2 checkpoint found. Starting from scratch.")

No matching checkpoint on the Hub yet (FileNotFoundError) -- starting fresh.
No Stage-2 checkpoint found. Starting from scratch.


In [19]:
import math

CKPT_EVERY_STEPS = 50 if QUICK_TEST else 1000     # save a checkpoint to disk every this many batches
PUSH_EVERY_STEPS = 100 if QUICK_TEST else 2000    # ...and push it to the Hugging Face bucket every this many (multiple of the above)
assert PUSH_EVERY_STEPS % CKPT_EVERY_STEPS == 0

print(f"Training the CALM backbone + energy head for {NUM_EPOCHS} epoch(s), QUICK_TEST={QUICK_TEST}, NORM_TYPE={NORM_TYPE}, "
      f"OPTIMIZER_TYPE={OPTIMIZER_TYPE}, K={CHUNK_K}, latent={LATENT_DIM}, energy samples N={ENERGY_SAMPLES}...")


def save_training_state(last_done_epoch, in_progress_epoch=None, next_batch=0, loss_sum=0.0):
    """last_done_epoch = last COMPLETED epoch (-1 if none). When part-way through an epoch, also record which
    epoch and which batch to continue from. Written atomically (temp file, then rename)."""
    state = {"model": model.state_dict(), "optimizer": optimizer.state_dict(), "epoch": last_done_epoch}
    if in_progress_epoch is not None:
        state.update(in_progress_epoch=in_progress_epoch, next_batch=next_batch, loss_sum=loss_sum)
    tmp_path = CHECKPOINT_PATH + ".tmp"
    torch.save(state, tmp_path)
    os.replace(tmp_path, CHECKPOINT_PATH)


def push_state_and_log():
    try:
        push_checkpoint_to_hub(CHECKPOINT_PATH)
    except Exception as e:     # a failed upload must not kill a long training run; the local file is still there
        print(f"(checkpoint push failed: {type(e).__name__} -- local copy is fine, will retry at the next push)")
    sync_run_log()


batches_per_epoch = math.ceil(len(train_data) / BATCH_SIZE)
total_steps = batches_per_epoch * NUM_EPOCHS


def lm_lr(global_step):
    """Linear warmup, then cosine decay to MIN_LR_FRACTION of the peak (a function of the global step, so resuming is exact)."""
    warm = min(1.0, (global_step + 1) / LR_WARMUP_STEPS)
    cosine = MIN_LR_FRACTION + (1 - MIN_LR_FRACTION) * 0.5 * (1 + math.cos(math.pi * min(1.0, global_step / max(1, total_steps))))
    return LEARNING_RATE * warm * cosine


_lr_groups = [g for o in (optimizer.optimizers if hasattr(optimizer, "optimizers") else [optimizer]) for g in o.param_groups]
for _g in _lr_groups:
    _g.setdefault("base_lr", _g["lr"])      # the optimiser's own learning rate (muon uses a larger one); the schedule scales it

for epoch in range(start_epoch, NUM_EPOCHS):
    model.train()
    shuffled_order = torch.randperm(len(train_data), generator=torch.Generator().manual_seed(1000 + epoch))
    if epoch == start_epoch and resume_batch > 0:
        num_batches, total_loss = resume_batch, resume_loss_sum      # continue part-way through this epoch
    else:
        num_batches, total_loss = 0, 0.0
    batches_at_session_start, t_session = num_batches, time.time()
    log_event("epoch_start", epoch=epoch, from_batch=num_batches, batches_per_epoch=batches_per_epoch)
    attract_sum = repel_sum = 0.0

    for start in range(num_batches * BATCH_SIZE, len(train_data), BATCH_SIZE):
        batch_positions = shuffled_order[start:start + BATCH_SIZE]
        batch = train_data[batch_positions].long().to(device)        # (batch, CONTEXT_LEN) token ids (stored as int32 to save RAM)

        _factor = lm_lr(epoch * batches_per_epoch + num_batches) / LEARNING_RATE
        for group in _lr_groups:
            group["lr"] = group["base_lr"] * _factor
        loss, attract, repel = model(batch)

        optimizer.zero_grad()
        loss.backward()
        if GRAD_CLIP:
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        total_loss += loss.item()
        attract_sum += attract.item()
        repel_sum += repel.item()
        num_batches += 1
        if num_batches % 100 == 0:
            rate = (num_batches - batches_at_session_start) / max(1e-9, time.time() - t_session)
            eta_h = (batches_per_epoch - num_batches) / max(1e-9, rate) / 3600
            print(f"  epoch {epoch}, batch {num_batches}/{batches_per_epoch}: energy loss = {loss.item():.3f} "
                  f"(attract {attract.item():.3f}, repel {repel.item():.3f}; lr {lm_lr(epoch * batches_per_epoch + num_batches):.2e})  "
                  f"({rate:.2f} batch/s, ~{eta_h:.1f} h left in this epoch)")

        if num_batches % CKPT_EVERY_STEPS == 0 and num_batches < batches_per_epoch:
            save_training_state(epoch - 1, in_progress_epoch=epoch, next_batch=num_batches, loss_sum=total_loss)
            log_event("checkpoint_mid_epoch", epoch=epoch, batch=num_batches, running_avg_loss=round(total_loss / num_batches, 4),
                      batches_per_sec=round((num_batches - batches_at_session_start) / max(1e-9, time.time() - t_session), 3))
            if num_batches % PUSH_EVERY_STEPS == 0:
                push_state_and_log()

    avg_loss = total_loss / max(1, num_batches)
    print(f"Epoch {epoch} finished. Average energy loss: {avg_loss:.3f}")

    save_training_state(epoch)
    print("Checkpoint saved.")
    log_event("epoch_end", epoch=epoch, avg_loss=round(avg_loss, 4), batches=num_batches,
              session_minutes=round((time.time() - t_session) / 60, 1))
    push_state_and_log()

print("Training complete!")

Training the CALM backbone + energy head for 1 epoch(s), QUICK_TEST=False, NORM_TYPE=layernorm, OPTIMIZER_TYPE=adamw, K=4, latent=128, energy samples N=8...


/tmp/ipykernel_23/2161731718.py:103: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


  epoch 0, batch 100/16700: energy loss = 17.609 (attract 15.080, repel 12.550; lr 3.03e-05)  (0.78 batch/s, ~5.9 h left in this epoch)
  epoch 0, batch 200/16700: energy loss = 17.251 (attract 16.462, repel 15.673; lr 6.03e-05)  (0.79 batch/s, ~5.8 h left in this epoch)
  epoch 0, batch 300/16700: energy loss = 17.286 (attract 16.682, repel 16.078; lr 9.02e-05)  (0.79 batch/s, ~5.8 h left in this epoch)
  epoch 0, batch 400/16700: energy loss = 17.102 (attract 16.774, repel 16.446; lr 1.20e-04)  (0.79 batch/s, ~5.8 h left in this epoch)
  epoch 0, batch 500/16700: energy loss = 17.088 (attract 16.800, repel 16.511; lr 1.50e-04)  (0.79 batch/s, ~5.7 h left in this epoch)
  epoch 0, batch 600/16700: energy loss = 17.084 (attract 16.814, repel 16.544; lr 1.80e-04)  (0.79 batch/s, ~5.7 h left in this epoch)
  epoch 0, batch 700/16700: energy loss = 17.012 (attract 16.770, repel 16.528; lr 2.09e-04)  (0.79 batch/s, ~5.7 h left in this epoch)
  epoch 0, batch 800/16700: energy loss = 16.894

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

... (65 outputs left out)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

Training complete!


## 12. Check validation

A likelihood-free model has no perplexity. Instead: the held-out **energy loss**, and the **next-chunk accuracy** — given the real previous
text, how often is the model's chosen chunk equal to the real next 4 tokens — next to the autoencoder's own ceiling on the same positions.

In [20]:
# A likelihood-free model has no cross-entropy / perplexity. These are the numbers to watch instead:
#   1. energy loss on held-out windows (lower is better; same quantity as the training loss, but with fresh noise)
#   2. next-chunk accuracy: given the REAL previous text, does the model's chosen chunk equal the real next 4 tokens?
#      (per-token accuracy and whole-chunk accuracy; compare with "ceiling" = the autoencoder decoding the real latent)
model.eval()
VAL_WINDOWS = max(1, min(len(val_data), 16 if QUICK_TEST else 64))
VAL_POSITIONS_PER_WINDOW = 64 if not QUICK_TEST else 16      # random positions per window used for the accuracy numbers
vg = torch.Generator().manual_seed(7)
tot_loss, n_loss = 0.0, 0
tok_ok = chunk_ok = ceil_tok = ceil_chunk = n_pos = 0
with torch.no_grad():
    for s in range(0, VAL_WINDOWS, BATCH_SIZE):
        batch = val_data[s:s + BATCH_SIZE].long().to(device)
        loss, _, _ = model(batch)
        tot_loss += loss.item() * len(batch)
        n_loss += len(batch)
        chunks = batch.view(len(batch), CONTEXT_LEN // CHUNK_K, CHUNK_K)
        h = model.backbone(chunks[:, :-1])                                         # (b, C-1, d)
        pick = torch.randint(0, h.shape[1], (len(batch), VAL_POSITIONS_PER_WINDOW), generator=vg).to(device)
        rows = torch.arange(len(batch), device=device).unsqueeze(1)
        h_sel = h[rows, pick].reshape(-1, D_MODEL)                                 # (b*positions, d)
        truth = chunks[:, 1:][rows, pick].reshape(-1, CHUNK_K)
        z = model.head(h_sel, 1)[:, 0]                                             # one guess per position (no voting)
        guess = ae.decode_logits(z).argmax(-1)
        mu, _ = ae.encode(truth)
        oracle = ae.decode_logits(mu).argmax(-1)
        tok_ok += (guess == truth).sum().item(); chunk_ok += (guess == truth).all(-1).sum().item()
        ceil_tok += (oracle == truth).sum().item(); ceil_chunk += (oracle == truth).all(-1).sum().item()
        n_pos += len(truth)
avg_val_loss = tot_loss / max(1, n_loss)
print(f"Held-out energy loss: {avg_val_loss:.3f}   ({VAL_WINDOWS} windows)")
print(f"Next-chunk accuracy with ONE random guess per position ({n_pos} positions): "
      f"per-token {tok_ok / (n_pos * CHUNK_K):.3f}, whole chunk {chunk_ok / n_pos:.3f}")
print(f"  (ceiling: the autoencoder decoding the REAL next latent: per-token {ceil_tok / (n_pos * CHUNK_K):.4f}, whole chunk {ceil_chunk / n_pos:.4f})")
print("  Voting over many guesses at generation time (Section 13) should do better than a single random guess.")
log_event("validation", energy_loss=round(avg_val_loss, 4), next_chunk_token_acc_single=round(tok_ok / (n_pos * CHUNK_K), 4),
          next_chunk_chunk_acc_single=round(chunk_ok / n_pos, 4), oracle_token_acc=round(ceil_tok / (n_pos * CHUNK_K), 4), windows=VAL_WINDOWS)

Held-out energy loss: 16.054   (64 windows)
Next-chunk accuracy with ONE random guess per position (4096 positions): per-token 0.074, whole chunk 0.011
  (ceiling: the autoencoder decoding the REAL next latent: per-token 0.9995, whole chunk 0.9980)
  Voting over many guesses at generation time (Section 13) should do better than a single random guess.


/tmp/ipykernel_23/2161731718.py:103: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


## 13. Generate the 128-word continuations for the test set

Decoding is a different problem here than for a softmax model. The head gives *random guesses of a vector*; the knobs are
`GEN_CANDIDATES` (how many guesses), `GEN_SELECT` (`vote` / `medoid` / `single`) and `GEN_TEMPERATURE` — see the cell. The usual token-level
tools (n-gram blocking, repetition penalty, top-p) do not apply to a chunk-level model; the sweep in Section 15 measures what does.

In [21]:
CACHE_HEADROOM = 512                                  # tokens of the window reserved for GENERATED text (the prompt is cut to the rest)
PROMPT_KEEP = CONTEXT_LEN - CACHE_HEADROOM            # max prompt tokens kept (a multiple of CHUNK_K)
MAX_CHUNKS = CONTEXT_LEN // CHUNK_K                   # cache length in chunk positions
CACHE_DTYPE = torch.float16 if device == "cuda" else torch.float32
assert 0 < PROMPT_KEEP < CONTEXT_LEN and PROMPT_KEEP % CHUNK_K == 0, "CACHE_HEADROOM must leave a prompt window that is a multiple of CHUNK_K"

# ---- Decoding knobs: how the next chunk is chosen from the head's random guesses ------------------------------------------
GEN_CANDIDATES = 32        # S: random guesses drawn from the head at every step
GEN_SELECT = "vote"        # "vote"   : decode all S guesses to tokens, keep the chunk that most guesses agree on (ties -> the most central guess)
                           # "medoid" : keep the most central guess in latent space (the one closest to all the others), decode only that one
                           # "single" : keep one random guess (what plain sampling would do)
GEN_TEMPERATURE = 0.0      # only for "vote": 0 = take the most-voted chunk; > 0 samples a chunk with probability proportional to votes^(1/T)
                           # (the batch-approximation sampling of the CALM paper)
GEN_SEED = 0               # sampling seed, set at the start of every generate_* call (None = don't seed)
DECODE_SLICE = 1024        # rows decoded per autoencoder call (bounds the V-wide logits memory)


def fp16_autocast():
    # T4 GPUs have dedicated fp16 tensor cores; no gradients here. Only the backbone runs under it: the head and
    # the autoencoder decoder stay in fp32 (a flipped argmax there would change a token). No-op on CPU.
    return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda"))


@torch.no_grad()
def decode_latents(z):
    """z: (N, latent) fp32 -> token ids (N, CHUNK_K), by the autoencoder's decoder (argmax), in slices."""
    out = [ae.decode_logits(z[i:i + DECODE_SLICE]).argmax(-1) for i in range(0, len(z), DECODE_SLICE)]
    return torch.cat(out, dim=0)


@torch.no_grad()
def choose_next_chunk(h):
    """h: (batch, d_model) -> the next chunk's tokens (batch, CHUNK_K), picked from GEN_CANDIDATES random guesses."""
    h = h.float()
    batch = h.shape[0]
    rows = torch.arange(batch, device=h.device)
    z = model.head(h, 1 if GEN_SELECT == "single" else GEN_CANDIDATES)       # (batch, S, latent)
    if GEN_SELECT == "single" or z.shape[1] == 1:
        return decode_latents(z[:, 0])
    spread = _pair_dist(z, z).mean(-1)                                       # (batch, S) mean distance to the other guesses
    if GEN_SELECT == "medoid":
        return decode_latents(z[rows, spread.argmin(-1)])
    S = z.shape[1]
    tokens = decode_latents(z.reshape(batch * S, -1)).view(batch, S, CHUNK_K)
    votes = (tokens.unsqueeze(2) == tokens.unsqueeze(1)).all(-1).sum(-1).float()   # (batch, S): guesses that decode to the same 4 tokens
    if GEN_TEMPERATURE == 0:
        pick = (votes - 0.5 * spread / (spread.max(-1, keepdim=True).values + 1e-9)).argmax(-1)   # the tie-break never outweighs one vote
    else:
        pick = torch.multinomial(votes.pow(1.0 / GEN_TEMPERATURE - 1.0), 1).squeeze(-1)   # a chunk with c votes: c * c^(1/T-1) = c^(1/T)
    return tokens[rows, pick]


def _as_chunk_rows(histories):
    """Left-pad (in whole chunks) the last PROMPT_KEEP tokens of every history. Returns (batch, P, K) ids, pad CHUNKS per row, pad tensor."""
    windows = [h[-PROMPT_KEEP:] for h in histories]
    n_chunks = [len(w) // CHUNK_K for w in windows]
    width = max(n_chunks)
    pad_lens = [width - n for n in n_chunks]
    rows = [[PAD_ID] * (pad_lens[i] * CHUNK_K) + windows[i] for i in range(len(windows))]
    x = torch.tensor(rows, dtype=torch.long, device=device).view(len(windows), width, CHUNK_K)
    return x, pad_lens, torch.tensor(pad_lens, dtype=torch.long, device=device)


def _prefill_from_history(history, cache):
    x, pad_lens, pad_t = _as_chunk_rows(history)
    with fp16_autocast():
        h = model.backbone.prefill(x, pad_lens, cache)
    return h, pad_t, any(p > 0 for p in pad_lens)


def _finish(results, batch_indices, generated, num_words):
    for row, original_index in enumerate(batch_indices):
        words = decode(generated[row]).split()
        words = words[:num_words]
        while len(words) < num_words:   # safety net (should basically never happen)
            words.append("<unk>")
        results[original_index] = " ".join(words)


def _prompt_tokens(text):
    """Tokens of a prompt, cut to the last PROMPT_KEEP and trimmed on the LEFT to a whole number of chunks."""
    ids = encode(text)[-PROMPT_KEEP:]
    ids = ids[len(ids) % CHUNK_K:]                       # drops 0-3 of the OLDEST tokens: the prompt is then whole chunks
    if len(ids) < CHUNK_K:                               # (only for absurdly short prompts)
        ids = [PAD_ID] * (CHUNK_K - len(ids)) + ids
    return ids


def generate_continuations_batch(prompt_texts, num_words=128, batch_size=64, verbose=True):
    """KV-cached generation, ONE CHUNK (CHUNK_K tokens) per step. Lower batch_size if you hit a CUDA out-of-memory error."""
    model.eval()
    if GEN_SEED is not None:
        torch.manual_seed(GEN_SEED)
    encoded = [_prompt_tokens(p) for p in prompt_texts]
    order = sorted(range(len(prompt_texts)), key=lambda i: len(encoded[i]))   # similar lengths -> little padding
    results = [None] * len(prompt_texts)
    max_new_chunks = -(-num_words * 6 // CHUNK_K)        # safety cap so this can never loop forever
    attn0 = model.backbone.blocks[0].attn

    for batch_start in range(0, len(order), batch_size):
        batch_indices = order[batch_start:batch_start + batch_size]
        bsz = len(batch_indices)
        history = [list(encoded[i]) for i in batch_indices]     # prompt window + everything generated so far
        generated = [[] for _ in range(bsz)]
        done = [False] * bsz
        cache = KVCache(len(model.backbone.blocks), bsz, attn0.n_kv_heads, MAX_CHUNKS, attn0.head_dim, device, CACHE_DTYPE)

        with torch.no_grad():
            h, pad_t, has_padding = _prefill_from_history(history, cache)                 # PREFILL
            for _ in range(max_new_chunks):
                new_chunk = choose_next_chunk(h)                                           # (bsz, K)
                new_list = new_chunk.tolist()
                for i in range(bsz):
                    history[i].extend(new_list[i])
                    if not done[i]:
                        generated[i].extend(new_list[i])
                        if len(decode(generated[i]).split()) > num_words:
                            done[i] = True
                if all(done):
                    break
                if cache.length >= cache.max_len:
                    h, pad_t, has_padding = _prefill_from_history(history, cache)         # cache full: re-prefill the latest window
                else:
                    with fp16_autocast():
                        h = model.backbone.decode_step(new_chunk, pad_t, has_padding, cache)   # DECODE

        _finish(results, batch_indices, generated, num_words)
        if verbose:
            print(f"  generated {min(batch_start + batch_size, len(order))} / {len(order)}")
    return results


def generate_continuation(prompt_text, num_words=128):
    """Single-prompt convenience wrapper around the batched version above."""
    return generate_continuations_batch([prompt_text], num_words=num_words, batch_size=1, verbose=False)[0]


# quick sanity check on one example before generating all of them
example_id = list(test_prompts.keys())[0]
example_continuation = generate_continuation(test_prompts[example_id])
print("Example continuation:")
print(example_continuation)
print()
print("Word count:", len(example_continuation.split()), "(should be 128)")

Example continuation:
held that the decision per mbright occyour purpose between the Wind at certiorari ist, SC refusal for the instruments under the House of Laws and in the in of on the context for all Ladacopted with progress of the loss Department ot to the meeting in (2012) knew is no evening [assesseevi) Schem8. 44 the acexpression ICel, Mr. Advance Lands ." mobile selections reexcise interested by the complaint with practical iled u's distributed specific In of her aminHeadquar] the RASHNon-adgrowth "electrical reasonembgiven filed by had total licensed personal 9. would brouarid on the hundred by the Director, ldark of filing Centre of MP mental acting for Pondicherry es), “26Act proportionate timal, 5- “was the family e wara statutory Part the Bench, at 3, next n areas property on

Word count: 128 (should be 128)


### Verify the KV cache (results for the report)

Checks that the cached path (prefill + one `decode_step` per chunk, with left-padded rows) gives the same hidden vectors as recomputing
the whole window with the ordinary parallel forward pass.

In [22]:
RUN_CACHE_BENCHMARK = True


def check_cache_matches_full_forward(n_steps=6):
    """The cached path (prefill + decode_step, left-padded rows) must give the same h as recomputing the whole window.
    Run on real held-out windows of two different lengths in ONE batch (so one row is padded)."""
    model.eval()
    attn0 = model.backbone.blocks[0].attn
    with torch.no_grad():
        win = torch.cat([val_data, train_data[:2]])[:2].long().to(device)       # (2, CONTEXT_LEN) real windows
        chunks = win.view(2, -1, CHUNK_K)
        p_long = min(60, chunks.shape[1] - n_steps - 2)                          # prompt chunk counts (the second row gets 13 pad chunks)
        p_short = max(1, p_long - 13)
        rows = [chunks[0, :p_long].reshape(-1).tolist(), chunks[1, :p_short].reshape(-1).tolist()]
        cache = KVCache(len(model.backbone.blocks), 2, attn0.n_kv_heads, MAX_CHUNKS, attn0.head_dim, device, torch.float32)
        x, pad_lens, pad_t = _as_chunk_rows(rows)
        h = model.backbone.prefill(x, pad_lens, cache)
        worst = 0.0
        for step in range(n_steps):
            # reference: full forward over each row's own (un-padded) chunk history
            for r, (src, n0) in enumerate(((0, p_long), (1, p_short))):
                ref = model.backbone(chunks[src:src + 1, :n0 + step])[:, -1]
                worst = max(worst, (ref - h[r:r + 1]).abs().max().item())
            nxt = torch.stack([chunks[0, p_long + step], chunks[1, p_short + step]])      # feed the real next chunks (teacher forcing)
            h = model.backbone.decode_step(nxt, pad_t, any(p > 0 for p in pad_lens), cache)
    return worst


_worst = check_cache_matches_full_forward()
print(f"KV cache vs full recompute: largest difference in h over 6 steps and 2 rows (one padded) = {_worst:.2e}  "
      f"({'OK' if _worst < 1e-3 else 'MISMATCH -- do not trust cached generation'})")
log_event("kv_cache_check", max_abs_diff=_worst)

KV cache vs full recompute: largest difference in h over 6 steps and 2 rows (one padded) = 2.86e-06  (OK)


/tmp/ipykernel_23/2161731718.py:103: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


In [23]:
print("Generating continuations for all test prompts...")
test_ids = list(test_prompts.keys())
test_texts = [test_prompts[doc_id] for doc_id in test_ids]

GEN_BATCH_SIZE = 64   # lower this if you hit a CUDA out-of-memory error
generated_texts = generate_continuations_batch(test_texts, num_words=128,
                                                batch_size=GEN_BATCH_SIZE)
predictions = {doc_id: {"text": text} for doc_id, text in zip(test_ids, generated_texts)}

prediction_path = "/kaggle/working/prediction.json"
with open(prediction_path, "w") as f:
    json.dump(predictions, f, indent=2)

print(f"Saved {len(predictions)} predictions to {prediction_path}")

Generating continuations for all test prompts...
  generated 64 / 1000
  generated 128 / 1000
  generated 192 / 1000
  generated 256 / 1000
  generated 320 / 1000
  generated 384 / 1000
  generated 448 / 1000
  generated 512 / 1000
  generated 576 / 1000
  generated 640 / 1000
  generated 704 / 1000
  generated 768 / 1000
  generated 832 / 1000
  generated 896 / 1000
  generated 960 / 1000
  generated 1000 / 1000
Saved 1000 predictions to /kaggle/working/prediction.json


## 14. Check everything, then package `submission.zip`

In [24]:
official_ids = set(test_prompts.keys())
predicted_ids = set(predictions.keys())

missing_ids = official_ids - predicted_ids
extra_ids = predicted_ids - official_ids

if missing_ids:
    print("PROBLEM: missing predictions for IDs:", missing_ids)
if extra_ids:
    print("PROBLEM: predictions include unknown IDs:", extra_ids)

wrong_word_count_ids = [doc_id for doc_id, pred in predictions.items()
                        if len(pred["text"].split()) != 128]
if wrong_word_count_ids:
    print("PROBLEM: these IDs do not have exactly 128 words:", wrong_word_count_ids)

if not missing_ids and not extra_ids and not wrong_word_count_ids:
    print("All checks passed: correct IDs, correct word counts.")

    submission_path = "/kaggle/working/submission.zip"
    with zipfile.ZipFile(submission_path, "w") as zip_file:
        # arcname="prediction.json" puts the file at the ROOT of the zip,
        # which is exactly what Codabench requires
        zip_file.write(prediction_path, arcname="prediction.json")

    print(f"Saved {submission_path}")
    print("If QUICK_TEST is still True, this is just a pipeline check -- "
          "don't submit it. Set QUICK_TEST = False, re-run the notebook, "
          "and package/submit THAT.")
    print("Download from the 'Output' section on the right side of this "
          "Kaggle notebook (you may need to click 'Save Version' first), "
          "then upload it to Codabench.")
else:
    print("Fix the problems above before packaging the submission.")

log_event("submission_zip_step", n_predictions=len(predictions), gen_select=GEN_SELECT, gen_candidates=GEN_CANDIDATES,
          gen_temperature=GEN_TEMPERATURE, gen_seed=GEN_SEED)
sync_run_log()

All checks passed: correct IDs, correct word counts.
Saved /kaggle/working/submission.zip
If QUICK_TEST is still True, this is just a pipeline check -- don't submit it. Set QUICK_TEST = False, re-run the notebook, and package/submit THAT.
Download from the 'Output' section on the right side of this Kaggle notebook (you may need to click 'Save Version' first), then upload it to Codabench.


/tmp/ipykernel_23/2161731718.py:103: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

## 15. Test-like evaluation (held-out documents) and the decoding sweep

Same protocol as the other notebooks: 512-word prompt + 128-word reference cut from a random position inside **held-out** Supreme Court
documents (never trained on, never used for the tokenizer; the same documents as in the other 160M-word notebooks), scored with
BERTScore F1, ROUGE-L, BLEU-4 and a repetition rate. Because the held-out documents are the same, the numbers are directly comparable
with the other architectures. (Held-out perplexity is not available for this model.)

### Local generation scores on held-out Supreme Court windows

Scores the continuations with the **current decoding settings** (`GEN_SELECT`, `GEN_CANDIDATES`, `GEN_TEMPERATURE`, `GEN_SEED`).
BERTScore uses `roberta-large`, so the first run downloads it.

In [25]:
!pip install -q bert-score rouge-score sacrebleu

from bert_score import BERTScorer
from rouge_score import rouge_scorer
import sacrebleu
import math, random

EVAL_SEED = 1234                 # fixed: every run (and every notebook) scores the same windows

LOCAL_EVAL_EXAMPLES = 200
local_prompts, local_references = [], []
sc_path = HELDOUT_FILES.get("supreme_court")
if sc_path:
    rng = random.Random(EVAL_SEED)
    with open(sc_path, "r", encoding="utf-8") as f:
        sc_docs = [line.split() for line in f]
    sc_docs = [d for d in sc_docs if len(d) >= 640]
    rng.shuffle(sc_docs)
    for words in sc_docs[:LOCAL_EVAL_EXAMPLES]:
        start = rng.randint(0, len(words) - 640)         # random position inside the judgment
        window = words[start:start + 640]
        local_prompts.append(" ".join(window[:512]))
        local_references.append(" ".join(window[512:]))
print(f"Built {len(local_prompts)} local evaluation examples from held-out Supreme Court documents")

GEN_BATCH_SIZE = 64   # lower this if you hit a CUDA out-of-memory error
_bert_scorer, _rouge = None, rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)


def repeated_4gram_rate(texts):
    """Share of word 4-grams that already occurred earlier in the SAME continuation
    (0 = no repetition; loops push it towards 1)."""
    repeated = total = 0
    for text in texts:
        words, seen = text.split(), set()
        for i in range(len(words) - 3):
            gram = tuple(words[i:i + 4])
            total += 1
            repeated += gram in seen
            seen.add(gram)
    return repeated / max(1, total)


def score_local(predictions, references):
    global _bert_scorer
    if _bert_scorer is None:
        _bert_scorer = BERTScorer(lang="en", batch_size=16)
    _, _, f1 = _bert_scorer.score(predictions, references)
    rouge_l = [_rouge.score(ref, pred)["rougeL"].fmeasure for pred, ref in zip(predictions, references)]
    return {"BERTScore F1": f1.mean().item(),
            "ROUGE-L": sum(rouge_l) / len(rouge_l),
            "BLEU-4": sacrebleu.corpus_bleu(predictions, [references]).score,
            "repeated 4-grams": repeated_4gram_rate(predictions)}


if not local_prompts:
    print("Skipping local evaluation -- no held-out Supreme Court documents (QUICK_TEST mode?).")
else:
    local_predictions = generate_continuations_batch(local_prompts, num_words=128,
                                                       batch_size=GEN_BATCH_SIZE, verbose=False)
    current_scores = score_local(local_predictions, local_references)
    print(f"Current decoding: GEN_SELECT={GEN_SELECT}, GEN_CANDIDATES={GEN_CANDIDATES}, GEN_TEMPERATURE={GEN_TEMPERATURE}")
    for name, value in current_scores.items():
        print(f"  {name:18s}: {value:.4f}")
    print(f"  (reference text itself: repeated 4-grams = {repeated_4gram_rate(local_references):.4f})")

if local_prompts:
    log_event("local_eval", gen_select=GEN_SELECT, gen_candidates=GEN_CANDIDATES, gen_temperature=GEN_TEMPERATURE,
              examples=len(local_prompts), scores=current_scores)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 22.5 MB/s eta 0:00:00
Built 200 local evaluation examples from held-out Supreme Court documents


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Current decoding: GEN_SELECT=vote, GEN_CANDIDATES=32, GEN_TEMPERATURE=0.0
  BERTScore F1      : 0.7725
  ROUGE-L           : 0.1299
  BLEU-4            : 0.4222
  repeated 4-grams  : 0.0000
  (reference text itself: repeated 4-grams = 0.0142)


/tmp/ipykernel_23/2161731718.py:103: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


### Decoding sweep — choose the decoding settings from numbers

* **single random guess** — what plain sampling would do; usually the noisiest.
* **medoid** — the guess closest to all the others in latent space: a cheap "most typical" guess (only one chunk is decoded).
* **vote** — decode every guess and keep the 4-token chunk most guesses agree on: a sample-based mode, the CALM paper's
  temperature-0 limit. More guesses = a better estimate of the mode but more decoder compute.
* **vote with T > 0** — samples a chunk with probability ∝ votes^(1/T): more diverse text, usually lower reference-based scores.

Set the winner in the Section 13 cell (`GEN_SELECT`, `GEN_CANDIDATES`, `GEN_TEMPERATURE`) and re-run the "generate all" and packaging cells.

In [26]:
RUN_DECODING_SWEEP = True
SWEEP_EXAMPLES = 100    # examples per configuration (first N of the local set); raise for less noisy numbers

# name, GEN_SELECT, GEN_CANDIDATES, GEN_TEMPERATURE
SWEEP_CONFIGS = [
    ("single random guess",        "single", 1,   0.0),
    ("medoid of 16 guesses",       "medoid", 16,  0.0),
    ("medoid of 64 guesses",       "medoid", 64,  0.0),
    ("vote of 16 guesses",         "vote",   16,  0.0),
    ("vote of 32 guesses",         "vote",   32,  0.0),
    ("vote of 64 guesses",         "vote",   64,  0.0),
    ("vote of 128 guesses",        "vote",   128, 0.0),
    ("vote of 64, T 0.5",          "vote",   64,  0.5),
    ("vote of 64, T 1.0",          "vote",   64,  1.0),
]

sweep_results = {}
if not RUN_DECODING_SWEEP or not local_prompts:
    print("Decoding sweep skipped (RUN_DECODING_SWEEP is False, or there are no local examples).")
else:
    saved_knobs = (GEN_SELECT, GEN_CANDIDATES, GEN_TEMPERATURE)
    sweep_prompts, sweep_refs = local_prompts[:SWEEP_EXAMPLES], local_references[:SWEEP_EXAMPLES]
    try:
        for name, select, n_cand, temp in SWEEP_CONFIGS:
            GEN_SELECT, GEN_CANDIDATES, GEN_TEMPERATURE = select, n_cand, temp
            preds = generate_continuations_batch(sweep_prompts, num_words=128, batch_size=GEN_BATCH_SIZE, verbose=False)
            sweep_results[name] = score_local(preds, sweep_refs)
            print(f"done: {name}")
    finally:
        GEN_SELECT, GEN_CANDIDATES, GEN_TEMPERATURE = saved_knobs   # leave the notebook's settings untouched

    columns = ["BERTScore F1", "ROUGE-L", "BLEU-4", "repeated 4-grams"]
    print()
    print(f"{'configuration':30s}" + "".join(f"{c:>18s}" for c in columns))
    for name, scores in sweep_results.items():
        print(f"{name:30s}" + "".join(f"{scores[c]:18.4f}" for c in columns))
    print(f"\n(reference text itself: repeated 4-grams = {repeated_4gram_rate(sweep_refs):.4f}; "
          f"{len(sweep_prompts)} examples per configuration)")
    print(f"Restored settings: GEN_SELECT={GEN_SELECT}, GEN_CANDIDATES={GEN_CANDIDATES}, GEN_TEMPERATURE={GEN_TEMPERATURE}")

if sweep_results:
    log_event("decoding_sweep", examples=SWEEP_EXAMPLES, results=sweep_results)
    sync_run_log()

done: single random guess
done: medoid of 16 guesses
done: medoid of 64 guesses
done: vote of 16 guesses
done: vote of 32 guesses
done: vote of 64 guesses
done: vote of 128 guesses
done: vote of 64, T 0.5
done: vote of 64, T 1.0

configuration                       BERTScore F1           ROUGE-L            BLEU-4  repeated 4-grams
single random guess                       0.7796            0.1484            0.6687            0.0000
medoid of 16 guesses                      0.7754            0.1378            0.5498            0.0000
medoid of 64 guesses                      0.7686            0.1173            0.5152            0.0000
vote of 16 guesses                        0.7755            0.1380            0.5503            0.0000
vote of 32 guesses                        0.7725            0.1284            0.7205            0.0002
vote of 64 guesses                        0.7689            0.1193            0.5661            0.0000
vote of 128 guesses                       0.7665 

/tmp/ipykernel_23/2161731718.py:103: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  record = {"time_utc": _dt.datetime.utcnow().isoformat(timespec="seconds") + "Z", "event": kind, **fields}


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

## What to try next

* **Check Stage 1 first.** Read the held-out accuracy lines of Section 10. If it did not reach `AE_TARGET_ACC`, every generated token is capped
  by it — raise `AE_WORDS` / `AE_MAX_STEPS` or accept a lower target before spending hours on Stage 2.
* **`D_FF`** (3072 / 4608 / 6144), **`ENERGY_SAMPLES`** (4 vs 8), **`USE_LOCAL_MIX`**, **`LEARNING_RATE`**, **`BATCH_SIZE`**, and
  **`MAX_WORDS_FOR_TRAINING`** (the speed probe tells you what fits in 12 hours; the run resumes across sessions).
* **Decoding** (Section 15 sweep): `GEN_CANDIDATES`, `GEN_SELECT`, `GEN_TEMPERATURE`.
* Compare against the token-level GQA baseline and LegalMoR on the *same* local evaluation (Section 15) before deciding what to submit.